# 1 Descargar Ventas y Visitas (aje-prd-analytics-artifacts-s3)

In [1]:
#!pip install 'awswrangler[redshift]'
#!pip install psycopg2-binary


!pip install --upgrade pip setuptools wheel
!pip install --upgrade pyarrow==17.0.0
!pip install "awswrangler[redshift]" --no-build-isolation
!pip install psycopg2-binary

In [2]:
# import psutil
# import time
# import threading
# import matplotlib.pyplot as plt

# # Variables globales
# mem_usage_gb = []
# mem_usage_percent = []
# cpu_usage_gb = []
# cpu_usage_percent = []
# timestamps = []
# running = True  # Variable de control

# # Función de monitoreo en segundo plano
# def monitor_resources():
#     global running
#     start_time = time.time()
    
#     while running:
#         # Obtener información del sistema
#         mem_info = psutil.virtual_memory()
#         cpu_percent = psutil.cpu_percent(interval=0.5)  # Medir CPU sin bloquear
#         cpu_used_gb = (cpu_percent / 100) * (psutil.virtual_memory().total / (1024**3))  # Estimación del uso de CPU en GB
        
#         # Guardar datos
#         timestamp = time.time() - start_time
#         mem_usage_gb.append(mem_info.used / (1024 ** 3))  # RAM usada en GB
#         mem_usage_percent.append(mem_info.percent)  # RAM en %
#         cpu_usage_gb.append(cpu_used_gb)  # CPU en GB
#         cpu_usage_percent.append(cpu_percent)  # CPU en %
#         timestamps.append(timestamp)
        
#         # Imprimir valores en tiempo real
#         # print(f"Tiempo: {timestamp:.2f}s | RAM: {mem_info.percent:.2f}% ({mem_usage_gb[-1]:.2f} GB) | CPU: {cpu_percent:.2f}% ({cpu_used_gb:.2f} GB)")
        
#         time.sleep(1)  # Medir cada 1 segundo

# # Iniciar monitoreo en segundo plano
# monitor_thread = threading.Thread(target=monitor_resources, daemon=True)
# monitor_thread.start()

# print("Monitoreo de CPU y RAM iniciado en segundo plano...")

In [3]:
import pandas as pd
import numpy as np
import redshift_connector
import awswrangler as wr
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import os
import boto3
import io
import pytz
import re
import gc

In [4]:
def comprobar_inputs():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/ecuador/"
    
    hoy = datetime.now(pytz.timezone("America/Lima")).date()
    errores = []

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    if "Contents" not in objetos:
        print("ERROR: No se encontraron archivos en la ruta especificada.")
        return

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        key = objeto["Key"]
        print(key)
        # Omitir "carpetas" en S3
        if key.endswith("/"):
            continue

        last_modified = objeto["LastModified"].date()
        size_kb = objeto["Size"] / 1024  # Convertir a KB

        # Verificar si el objeto tiene contenido real (FULL_OBJECT)
        if objeto["Size"] == 0:
            errores.append(f"ERROR: El archivo {key} está vacío.")

        # Verificar si el archivo ha sido modificado hoy
        if last_modified != hoy:
            errores.append(f"ERROR: El archivo {key} no ha sido modificado hoy ({hoy}), su ultima fecha de modificacion fue {last_modified}.")

        # Verificar si el tamaño del archivo es menor a 1 KB
        if size_kb < 1:
            errores.append(f"ALERTA: El archivo {key} tiene un tamaño menor a 1 KB ({size_kb:.2f} KB).")

    # Mostrar los errores y lanzar una excepción si es necesario
    if errores:
        for error in errores:
            print(error)
        raise ValueError("Se encontraron problemas con los archivos en S3.")
    else:
        print("Todo bien :D")

In [5]:
comprobar_inputs()

pedido_sugerido/data-v1/ecuador/
pedido_sugerido/data-v1/ecuador/D_stock_ec.csv
pedido_sugerido/data-v1/ecuador/EC_maestro_productos.csv
pedido_sugerido/data-v1/ecuador/maestro_productos_ecuador000
pedido_sugerido/data-v1/ecuador/ventas_ecuador000
pedido_sugerido/data-v1/ecuador/ventas_ecuador_tomorrow.csv
pedido_sugerido/data-v1/ecuador/visitas_ecuador000
Todo bien :D


## 1.0 Descargar Maestro de productos

In [6]:
# PARA 2024
query = f"""select * from 
    comercial_ecuador.dim_producto
    where estado='A' and instancia='EC';
    """

con = wr.data_api.redshift.connect(
    cluster_id="dwh-cloud-storage-salesforce-prod",
    database="dwh_prod",
    db_user="dwhuser",
)
maestro_prod = wr.data_api.rds.read_sql_query(query, con)

In [7]:
maestro_prod.head()

,id_producto,instancia,desc_compania,cod_articulo_magic,desc_articulo,desc_articulo_corto,cod_linea,desc_linea,cod_familia,desc_familia,...,flg_tipo_composicion,flg_linea,paquete,flg_preventa,flgabastec,flgskuplan,flg_explosion,fecha_creacion,cod_unidad_negocio,desc_unidad_negocio
0,EC|0030|60968,EC,AJEMEX ...,60968,PREFORMA DE 17.15 GRS CRISTAL AGUA PT,PREFORMA DE 17.15 GRS CRISTAL,01,PRODUCTO TERMINADO,015,PREFORMAS,...,C,Te,1,N,OP,N,N,2015-06-05,,None
1,EC|0030|60969,EC,AJEMEX ...,60969,PREFORMA DE 17.15 GRS CRISTAL CON UV PT,PREFORMA DE 17.15 GRS CRISTAL,01,PRODUCTO TERMINADO,015,PREFORMAS,...,C,Te,1,N,OP,N,N,2015-06-05,,None
2,EC|0030|60970,EC,AJEMEX ...,60970,PREFORMA DE 17.15 GRS CRISTAL PT,PREFORMA DE 17.15 GRS CRISTAL,01,PRODUCTO TERMINADO,015,PREFORMAS,...,C,Te,1,N,OP,N,N,2015-06-05,,None
3,EC|0030|60971,EC,AJEMEX ...,60971,PREFORMA DE 17.15 GRS AZUL PT,PREFORMA DE 17.15 GRS AZUL PT,01,PRODUCTO TERMINADO,015,PREFORMAS,...,C,Te,1,N,OP,N,N,2015-06-05,,None
4,EC|0030|60972,EC,AJEMEX ...,60972,PREFORMA DE 17.15 GRS AZUL AGUA PT,PREFORMA DE 17.15 GRS AZUL AG,01,PRODUCTO TERMINADO,015,PREFORMAS,...,C,Te,1,N,OP,N,N,2015-06-05,,None


In [8]:
maestro_prod.groupby(["cod_articulo_magic", "desc_articulo"]).head(1)[["cod_articulo_magic", "desc_articulo"]].to_csv("Input/EC_maestro_productos.csv", index=False)

In [9]:
# Fecha de hoy
hoy = datetime.today()

# Fecha de hace 12 meses
hace_12_meses = hoy - timedelta(days=365)

# Formato año-mes-dia
fecha_i = hoy.strftime("%Y-%m-%d")
fecha_f = hace_12_meses.strftime("%Y-%m-%d")
print(fecha_i, fecha_f)

2026-03-30 2025-03-30


In [10]:
# 1403, 5617, 1304, # Piloto
rutas_test = [1008,1108,1208,1308,1408,1608,2808,1708,1808, # Ambato
              1908,2008,2108,2208,2408,
              1003,1103,1203,1303,1403,1503,1603,1703,4403, # Guayaquil norte
              1803,1903,2003,2103,2203,2303,2403,2503,2603,
              2703,2803,2903,3003,3103,3203,3303,3403,4703,
              4903,5203,5603,5703,
              1017,1117,1217,1317,1417,1517,1617,1717,1817, # Guayaquil Sur
              1917,2017,2117,2217,2317,2417,2517,2617,2717,
              2817,2917,3017,3117,3217,3317,3417,3517,3617,
              3717,3817,3917,4017,4117,4217,4317,4417,4517,
              4617,4717,4817,4917,5517,5617,5717,
              1006,1106,1206,1306,1406,1506,1606,1706,1806, # Machala
              1906,2006,2106,2206,2306,3806,4106,2406,2506,
              2606,2706,2806,2906,3006,3106,
              1007,1307,1507,1807,1907,2007,2107,2307,2407, # Manta
              2507,2607,2707,2807,3607,1107,1207,1407,1607,
              2207,3507,3707,3307,3107,
              1004,1104,1204,1304,1404,1504,1604,1704,1804, # Quito norte
              1904,2004,2104,2204,2304,2404,2504,2604,2704,
              2804,2904,3004,3104,3204,3304,3404,3504,3604,
              4504,4604,4704,4804,4904,5104,5204,5404,
              1011,1211,1311,1411,1611,1711,1811,1911,1511, # Quito sur
              2011,2111,2211,2511,2711,2811,4711,2311,2411,
              2611,2911,3411,3511,3611,3911,3011,3111,3211,
              3311,3711,3811,4011,4811,
              1009,1209,1609,1909,2009,2609,2709,2809,1109, # Santo Domingo
              1309,1409,1709,1809,2109,2209]
              
print(rutas_test)

[1008, 1108, 1208, 1308, 1408, 1608, 2808, 1708, 1808, 1908, 2008, 2108, 2208, 2408, 1003, 1103, 1203, 1303, 1403, 1503, 1603, 1703, 4403, 1803, 1903, 2003, 2103, 2203, 2303, 2403, 2503, 2603, 2703, 2803, 2903, 3003, 3103, 3203, 3303, 3403, 4703, 4903, 5203, 5603, 5703, 1017, 1117, 1217, 1317, 1417, 1517, 1617, 1717, 1817, 1917, 2017, 2117, 2217, 2317, 2417, 2517, 2617, 2717, 2817, 2917, 3017, 3117, 3217, 3317, 3417, 3517, 3617, 3717, 3817, 3917, 4017, 4117, 4217, 4317, 4417, 4517, 4617, 4717, 4817, 4917, 5517, 5617, 5717, 1006, 1106, 1206, 1306, 1406, 1506, 1606, 1706, 1806, 1906, 2006, 2106, 2206, 2306, 3806, 4106, 2406, 2506, 2606, 2706, 2806, 2906, 3006, 3106, 1007, 1307, 1507, 1807, 1907, 2007, 2107, 2307, 2407, 2507, 2607, 2707, 2807, 3607, 1107, 1207, 1407, 1607, 2207, 3507, 3707, 3307, 3107, 1004, 1104, 1204, 1304, 1404, 1504, 1604, 1704, 1804, 1904, 2004, 2104, 2204, 2304, 2404, 2504, 2604, 2704, 2804, 2904, 3004, 3104, 3204, 3304, 3404, 3504, 3604, 4504, 4604, 4704, 4804, 490

In [11]:
len(rutas_test)

217

In [12]:
clientes_ruta_test = []

In [13]:
def descargar_visitas():
    # Guardar clientes de las rutas
    global clientes_ruta_test

    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/ecuador/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] == "visitas_ecuador000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Descargar el archivo en memoria
            response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
            content = response["Body"].read()

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "visitas_ecuador000":
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    # (df["compania__c"] == 90)
                    # & (df["cod_ruta"].isin(rutas_test))
                    (df["cod_ruta"].isin(rutas_test))
                    # & (df.codigo_canal__c == 2)
                ].reset_index(drop=True)
                # Guardamos estos clientes para descargarlos en ventas (por si cambiaron de ruta)
                clientes_ruta_test = df["codigo_cliente__c"].unique()
                nombre_csv = f"Input/{nombre_archivo}.parquet"
                df["personalizacion"] = df["personalizacion"].astype(str)
                df.to_parquet(nombre_csv, index=False)
            else:
                continue

In [14]:
%%time
descargar_visitas()

{'ResponseMetadata': {'RequestId': 'VQ9GC29T7XEF9V36', 'HostId': 'TUm4TBIEe1DzA+d83c26hTcCI8T7opqRkEJ7cyFdUSsrunZgNi0N62YHHMgDoMq/F556s/a2fn55E1ty7UN7ikn4Ub1iLPb9', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'TUm4TBIEe1DzA+d83c26hTcCI8T7opqRkEJ7cyFdUSsrunZgNi0N62YHHMgDoMq/F556s/a2fn55E1ty7UN7ikn4Ub1iLPb9', 'x-amz-request-id': 'VQ9GC29T7XEF9V36', 'date': 'Mon, 30 Mar 2026 20:23:18 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/ecuador/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 32, 4, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/ecuador/D_stock_ec.csv', 'LastModified': datetime.datetime(2026, 3, 30, 14, 3, 33, tzinfo=tzlocal()), 'ETag': '"3735f313af8a36807f4ef593c3d9a675"', 'ChecksumAlgorithm': ['CRC32'], '

In [15]:
def descargar_ventas():
    # Conectarse a S3
    s3 = boto3.client("s3")
    bucket_name = "aje-prd-analytics-artifacts-s3"
    prefix = "pedido_sugerido/data-v1/ecuador/"

    # Listar objetos en la ruta de S3
    objetos = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

    print(objetos)
    
    final_csv = pd.DataFrame()

    # Iterar sobre los objetos
    for objeto in objetos["Contents"]:
        if objeto["Size"] > 0 and objeto["Key"].split("/")[-1] != "visitas_ecuador000":
            # Obtener el nombre del objeto y descargarlo
            nombre_archivo = objeto["Key"].split("/")[-1]

            # Filtrar y guardar el archivo si corresponde
            if nombre_archivo == "ventas_ecuador000":
                # Descargar el archivo en memoria
                response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
                content = response["Body"].read()
                # Convertir los bytes a DataFrame de Pandas
                df = pd.read_csv(io.BytesIO(content), sep=";")
                df = df[
                    (df["cod_compania"] == 90)
                    & (
                        (df["cod_ruta"].isin(rutas_test))
                        | (df["cod_cliente"].isin(clientes_ruta_test))
                    )
                ].reset_index(drop=True)[['id_cliente', 'id_sucursal', 'id_producto',
       'fecha_liquidacion',"cod_ruta","cod_modulo",
       'cod_zona', 'cant_cajafisicavta', 'cant_cajaunitvta','imp_netovta',
       'cod_compania', 'desc_compania', 'cod_sucursal',
       'desc_sucursal', 'cod_pais', 'fecha_creacion_cliente', 'cod_cliente',
       'desc_marca', 'desc_formato', 'desc_categoria', 'cod_giro',
       'cod_subgiro', 'desc_giro', 'desc_subgiro', 'fecha_proceso']]
                final_csv = pd.concat([final_csv, df], ignore_index=True)
            elif nombre_archivo == "ventas_ecuador001":
                # # Descargar el archivo en memori
                # response = s3.get_object(Bucket=bucket_name, Key=objeto["Key"])
                # content = response["Body"].read()
                # # Convertir los bytes a DataFrame de Pandas
                # df = pd.read_csv(io.BytesIO(content), sep=";")
                # df = df[
                #     (df["cod_compania"] == 90)
                #     & (
                #         (df["cod_ruta"].isin(rutas_test))
                #         | (df["cod_cliente"].isin(clientes_ruta_test))
                #     )
                # ].reset_index(drop=True)
                # final_csv = pd.concat([final_csv, df], ignore_index=True)
                continue
            else:
                continue

    nombre_csv = f"Input/ventas_ec_all.parquet"
    final_csv.to_parquet(nombre_csv, index=False)

In [16]:
%%time
descargar_ventas()

{'ResponseMetadata': {'RequestId': 'QAD9TK3R6AHKD1NQ', 'HostId': '9nfB5EqWomViPV5gQDVrMx73WvIL/WN6+DhoR1SbsVUICpHO1ZYQn+FVbujCdfZnCOIH1AkZ0eAH7oM+tcYf9nV+u5VilKz8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '9nfB5EqWomViPV5gQDVrMx73WvIL/WN6+DhoR1SbsVUICpHO1ZYQn+FVbujCdfZnCOIH1AkZ0eAH7oM+tcYf9nV+u5VilKz8', 'x-amz-request-id': 'QAD9TK3R6AHKD1NQ', 'date': 'Mon, 30 Mar 2026 20:23:19 GMT', 'x-amz-bucket-region': 'us-east-2', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': False, 'Contents': [{'Key': 'pedido_sugerido/data-v1/ecuador/', 'LastModified': datetime.datetime(2024, 2, 8, 8, 32, 4, tzinfo=tzlocal()), 'ETag': '"d41d8cd98f00b204e9800998ecf8427e"', 'Size': 0, 'StorageClass': 'STANDARD'}, {'Key': 'pedido_sugerido/data-v1/ecuador/D_stock_ec.csv', 'LastModified': datetime.datetime(2026, 3, 30, 14, 3, 33, tzinfo=tzlocal()), 'ETag': '"3735f313af8a36807f4ef593c3d9a675"', 'ChecksumAlgorithm': ['CRC32'], '

## 1.1 Juntar VENTAS y VISITAS

In [17]:
pan_ventas = pd.read_parquet("Input/ventas_ec_all.parquet")
pan_visitas = pd.read_parquet("Input/visitas_ecuador000.parquet")

In [18]:
pan_visitas = pan_visitas[(pan_visitas.codigo_canal__c==2)&(pan_visitas.compania__c=="0090")].reset_index(drop=True)

In [19]:
pan_ventas["cod_articulo_magic"] = pan_ventas["id_producto"].str.split("|").str[-1]
pan_ventas["cod_articulo_magic"] = pan_ventas["cod_articulo_magic"].astype(int)

In [20]:
pan_ventas.shape

(10579712, 26)

In [21]:
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print(hoy)
# Filtrar solo las filas con la fecha de hoy
pan_ventas = pan_ventas[pan_ventas['fecha_proceso'] == hoy]

20260330


In [22]:
pan_ventas.shape

(10579712, 26)

In [23]:
# Obtener la fecha y año actual
current_date = datetime.now(pytz.timezone("America/Lima"))
# Formatear la fecha en el formato "YYYYMM"
formatted_date = current_date.strftime("%Y-%m-%d")
formatted_date

'2026-03-30'

In [24]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Ecuador/Input/visitas_ecuador000_{formatted_date}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(pan_visitas, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Ecuador/Input/visitas_ecuador000_2026-03-30.csv'],
 'partitions_values': {}}

In [25]:
#pan_ventas[["cod_compania","cod_articulo_magic","cod_portafolio_producto"]].drop_duplicates().sort_values("cod_articulo_magic").reset_index(drop=True).to_excel("D_PS_EC_compania_producto_portafolio.xlsx")

In [26]:
# PAN VENTAS
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_ventas["id_cliente"] = (
    "EC"
    + "|"
    + pan_ventas["cod_compania"].astype(str)
    + "|"
    + pan_ventas["cod_cliente"].astype(str)
)

# PAN VISITAS
pan_visitas["compania__c"] = (
    pan_visitas["compania__c"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)
pan_visitas["id_cliente"] = (
    "EC"
    + "|"
    + pan_visitas["compania__c"].astype(str)
    + "|"
    + pan_visitas["codigo_cliente__c"].astype(str)
)

In [27]:
# Eliminar Duplicados y quedarse con la ultima visita más reciente
visita_default_semana_atras = (datetime.now(pytz.timezone("America/Lima")) - timedelta(days=7)).strftime("%Y-%m-%d")
pan_visitas["ultima_visita"] = pan_visitas["ultima_visita"].fillna(
    visita_default_semana_atras
)
pan_visitas = (
    pan_visitas.sort_values(["id_cliente", "ultima_visita"], ascending=False)
    .groupby("id_cliente")
    .head(1)
)

In [28]:
# Combina ventas y visitas (nos quedamos con ruta y módulo de visitas si es nulo, usamos el valor de ventas)
df_merged = pd.merge(
    pan_ventas,
    pan_visitas[
        [
            "id_cliente",
            "dias_de_visita__c",
            "periodo_de_visita__c",
            "ultima_visita",
            "cod_ruta",
            "cod_modulo",
            "eje_potencial__c",
        ]
    ],
    on="id_cliente",
    # how='outer',  # Este es un merge 'outer', para conservar todos los clientes de ambos DataFrames.
    how="inner",
    suffixes=(
        "_df1",
        "_df2",
    ),  # Añadimos sufijos para diferenciar las columnas que se solapan
)

# Reemplazar los valores nulos de 'cod_ruta_df2' con los valores de 'cod_ruta_df1', y viceversa
# Si 'cod_ruta_df2' es nulo, se usará el valor de 'cod_ruta_df1'. Si ambos son nulos, se mantendrá nulo.
df_merged["cod_ruta"] = df_merged["cod_ruta_df2"].combine_first(
    df_merged["cod_ruta_df1"]
)
df_merged["cod_modulo"] = df_merged["cod_modulo_df2"].combine_first(
    df_merged["cod_modulo_df1"]
)

# Eliminar las columnas innecesarias ('cod_ruta_df1', 'cod_ruta_df2', 'cod_modulo_df1', 'cod_modulo_df2')
df_merged = df_merged.drop(
    columns=["cod_ruta_df1", "cod_ruta_df2", "cod_modulo_df1", "cod_modulo_df2"]
)

In [29]:
pan_ventas = df_merged.copy()
# Liberar memoria del df_merged
df_merged = None 
gc.collect()

1565

In [30]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(str)
pan_ventas["cod_ruta"] = pan_ventas["cod_ruta"].astype(int)
pan_ventas["cod_modulo"] = pan_ventas["cod_modulo"].astype(int)

In [31]:
# Al cruzar visita con ventas. hay clientes de. visita que no tienen informacion de venta, esto se remplaza a nivel de ruta.
# Este df guarda clientes con visitas y sin ventas para luego recomendarlos por separado
df_clientes_con_visitas_sin_ventas = pd.DataFrame()
for ruta in pan_ventas["cod_ruta"].unique():
    temp_df = pan_ventas[pan_ventas["cod_ruta"] == ruta]
    # Si la ruta entera no tiene ninguna venta, se guarda en un dataframe aparte
    if len(temp_df) == (temp_df["cod_articulo_magic"].isnull().sum()):
        df_clientes_con_visitas_sin_ventas = pd.concat(
            [df_clientes_con_visitas_sin_ventas, temp_df], axis=0
        ).reset_index(drop=True)

In [32]:
pan_ventas.groupby("cod_sucursal").agg(
    {"cod_articulo_magic": "nunique", "id_cliente": "nunique", "cod_ruta": "nunique"}
)

,cod_articulo_magic,id_cliente,cod_ruta
cod_sucursal,,,
3,133,7625,32
4,132,10401,35
6,140,6326,24
7,130,5486,22
8,124,3628,14
9,122,4159,15
11,121,8807,32
17,128,10548,43


# 2 Ajustar DF de clientes: Se filtrará a los clientes que tienen visita al día siguiente

## 2.2 Combinar Ventas con Subgiro, Descripcion de productos, Segmento del cliente y Fecha de creacion

In [33]:
import pandas as pd

# CAMBIAR DEPORADE POR SPORADE
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].str.strip()
pan_ventas["desc_marca"] = pan_ventas["desc_marca"].replace({"DEPORADE": "SPORADE"})

pan_prod = pd.read_csv("Input/EC_maestro_productos.csv")

# Crear ID_CLIENTE
# PAN_VENTAS
pan_ventas["cod_sucursal"] = pan_ventas["cod_sucursal"].astype(str)

# PAN SEGMENTOS
pan_ventas = pan_ventas.rename(columns={"eje_potencial__c": "new_segment"})
# mapear valores
mapping = {
    "S1": "BLINDAR",
    "S2": "DESARROLLAR",
    "S4": "MANTENER",
    "S5": "OPTIMIZAR",
}

pan_ventas["new_segment"] = pan_ventas["new_segment"].map(mapping)
pan_ventas["new_segment"] = pan_ventas["new_segment"].fillna("OPTIMIZAR")

In [34]:
pan_ventas = pd.merge(
    pan_ventas,
    pan_prod[["cod_articulo_magic", "desc_articulo"]],
    how="left",
    on="cod_articulo_magic",
)

pan_ventas["mes"] = pd.to_datetime(pan_ventas["fecha_liquidacion"]).dt.strftime(
    "%Y-%m-01"
)

In [35]:
pan_ventas.to_parquet("Processed/ventas_ecuador_12m.parquet", index=False)

## 2.4 Filtrar clientes a visitar mañana

In [36]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [37]:
pan_ventas = pd.read_parquet("Processed/ventas_ecuador_12m.parquet")

In [38]:
pan_ventas.id_cliente.nunique()

56979

In [39]:
pan_ventas.dias_de_visita__c.unique()

array(['1;4', '3;6', '2;5', '2', '4', '1', '3', '5', '6', '1;2;3;4;5;6;7',
       '1;4;5', '3;7'], dtype=object)

In [40]:
pan_ventas.groupby("dias_de_visita__c").id_cliente.nunique()

dias_de_visita__c
1                  294
1;2;3;4;5;6;7       15
1;4              18293
1;4;5                2
2                  261
2;5              18302
3                   88
3;6              19114
3;7                  2
4                  280
5                  227
6                  101
Name: id_cliente, dtype: int64

In [41]:
data_test = (
    pan_ventas[
        ["id_cliente", "dias_de_visita__c", "periodo_de_visita__c", "ultima_visita"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)
data_test["ultima_visita"] = pd.to_datetime(
    data_test["ultima_visita"], format="%Y-%m-%d"
)
# Calcula la diferencia en días entre la fecha de cada fila y la fecha de hoy
fecha_actual = datetime.now() + timedelta(days=0)
data_test["dias_pasados"] = (fecha_actual - data_test["ultima_visita"]).dt.days
fecha_actual

datetime.datetime(2026, 3, 30, 20, 29, 7, 897956)

In [42]:
# Obtener el día de la semana actual (1 para lunes, 2 para martes, ..., 7 para domingo)
dia_actual = datetime.now(pytz.timezone("America/Lima")).weekday() + 1

# Si hoy es domingo (7), el día siguiente es lunes (1), de lo contrario, es el siguiente día al actual
if dia_actual == 6:
    dia_siguiente = 7
    # dia_siguiente=6
else:
    dia_siguiente = (dia_actual + 1) % 7
    # dia_siguiente = (dia_actual ) % 7

# Filtrar los clientes que serán visitados mañana
clientes_a_visitar_manana = data_test[
    data_test["dias_de_visita__c"]
    .astype(str)
    .apply(lambda x: str(dia_siguiente) in x.split(";"))
].reset_index(drop=True)
print("dia_actual", dia_actual)
print("dia_siguiente", dia_siguiente)

dia_actual 1
dia_siguiente 2


In [43]:
# Definir condiciones de filtro
condicion_f1 = clientes_a_visitar_manana["periodo_de_visita__c"] == "F1"
condicion_f2 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F2") & (
    clientes_a_visitar_manana["dias_pasados"] > 13
)
condicion_f3 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F3") & (
    clientes_a_visitar_manana["dias_pasados"] > 20
)
condicion_f4 = (clientes_a_visitar_manana["periodo_de_visita__c"] == "F4") & (
    clientes_a_visitar_manana["dias_pasados"] > 27
)

# Aplicar las condiciones de filtro
clientes_a_visitar_manana = clientes_a_visitar_manana[
    condicion_f1 | condicion_f2 | condicion_f3 | condicion_f4
].reset_index(drop=True)

In [44]:
clientes_a_visitar_manana.isnull().sum()

id_cliente              0
dias_de_visita__c       0
periodo_de_visita__c    0
ultima_visita           0
dias_pasados            0
dtype: int64

In [45]:
clientes_a_visitar_manana.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [46]:
clientes_a_visitar_manana.dias_de_visita__c.unique()

array(['2;5', '2', '1;2;3;4;5;6;7'], dtype=object)

In [47]:
pan_ventas.groupby("cod_ruta")["dias_de_visita__c"].unique().reset_index()

,cod_ruta,dias_de_visita__c
0,1003,"[3;6, 2;5, 1;4]"
1,1004,"[3;6, 1;4, 2;5]"
2,1006,"[3;6, 1;4, 2;5]"
3,1007,"[3;6, 2;5, 1;4, 5, 2, 1, 4]"
4,1008,"[3;6, 1;4, 2;5]"
...,...,...
211,5517,"[1;4, 3;6, 2;5]"
212,5603,"[1;4, 2;5, 3;6]"
213,5617,"[3;6, 2;5, 1;4]"
214,5703,"[1;4, 2;5, 3;6]"


In [48]:
pan_ventas = pan_ventas[
    pan_ventas["id_cliente"].isin(clientes_a_visitar_manana["id_cliente"])
].reset_index(drop=True)

In [49]:
pan_ventas.periodo_de_visita__c.unique()

array(['F1'], dtype=object)

In [50]:
pan_ventas.dias_de_visita__c.unique()

array(['2;5', '2', '1;2;3;4;5;6;7'], dtype=object)

In [51]:
pan_ventas.id_cliente.nunique()

18578

In [52]:
# pan_ventas[["id_cliente", 'cod_sucursal', 'desc_sucursal','dias_de_visita__c',
#        'periodo_de_visita__c', 'ultima_visita', 'cod_ruta', 'cod_modulo']].drop_duplicates().to_csv("PS_EC_clientes_2024-10-31.csv",index=False)

In [53]:
pan_ventas.to_parquet("Processed/ecuador_ventas_manana.parquet", index=False)

# 3. Pre Procesamiento

In [54]:
import pandas as pd
import numpy as np
from datetime import datetime

In [55]:
df_ventas = pd.read_parquet("Processed/ecuador_ventas_manana.parquet")

In [56]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(
    df_ventas["fecha_liquidacion"], format="%Y-%m-%d"
)
df_ventas["desc_marca"] = df_ventas["desc_marca"].str.strip()
df_ventas["desc_categoria"] = df_ventas["desc_categoria"].str.strip()

## 3.1 Mapeo de Pesos por Giro
Los pesos representan el orden en el que se recomendara la marca para cada cliente dependiendo del Giro

In [57]:
# Mapeo de pesos para ordenar las recomendaciones finales segun su importancia por giro
mapeo_pesos = {}
mapeo_diccionario = {}

In [58]:
df_ventas.desc_marca.unique()

array(['BIG', 'CIFRUT', 'CIELO', 'PULP', 'VOLT', 'D GUSSTO', 'SPORADE',
       'DESS', 'ORO', 'NUTRIDOG GOLD', 'BLING', 'DILYTE', 'KR', 'VIDA'],
      dtype=object)

In [59]:
for giro_v in df_ventas["desc_subgiro"].unique():
    temp = df_ventas[(df_ventas["desc_subgiro"] == giro_v)]
    ranks = temp.groupby("desc_categoria")["cant_cajafisicavta"].sum().reset_index()
    ranks.columns = ["index","desc_categoria"]
    ranks = ranks.sort_values(by="desc_categoria", ascending=False)
    if len(list(ranks["desc_categoria"])) <= 5:
        ranks["Ranking"] = range(1, len(ranks) + 1)
    elif len(list(ranks["desc_categoria"])) > 5:
        a = list(ranks["desc_categoria"])
        b = [1, 1, 2, 2]
        # Calculamos el multiplicador para el mapeo de pesos segun la varianza
        if np.std(a) / np.mean(a) <= 1.2:
            multiplicador = 4
        else:
            multiplicador = 2
        if len(a) > 5:
            for i in range(4, len(a)):
                if a[3] <= a[i] * multiplicador:
                    b.append(3)
                else:
                    b.append(3 + i)
        ranks["Ranking"] = b

    print("*" * 20)
    print("GIRO: ", giro_v)
    print("Categorias: ", list(ranks["index"]))
    print("Counts: ", list(ranks["desc_categoria"]))
    print(ranks.set_index("index")["Ranking"].to_dict())
    mapeo_diccionario[giro_v] = ranks.set_index("index")["Ranking"].to_dict()

********************
GIRO:  BODEGAS, TIENDAS
Categorias:  ['GASEOSAS', 'BEBIDAS DEPORTIVAS', 'AGUA', 'NECTAR', 'CITRUS', 'ENERGIZANTE', 'DETERGENTE', 'CAFE', 'MASCOTA', 'CONSERVA DE PESCADO', 'DESINFECTANTE', 'ACEITE', 'SUERO', 'GELATINA', 'LAVAVAJILLAS', 'SUAVIZANTES', 'MODIFICADORES LACTEOS', 'ARROZ', 'AVENA', 'PASTAS', 'CEREALES', 'SNACKS']
Counts:  [920179.9999999991, 786649.5833333333, 536262.25, 496801.95833333326, 432864.31666666665, 174720.41666666666, 80546.83333333323, 36578.33333333332, 22263.0, 19996.541666666497, 16723.6666666665, 14036.458333333312, 7754.333333333054, 7109.333333333331, 2895.0, 2757.4999999999995, 2466.708333333332, 873.46, 729.5583333333333, 664.1, 139.3388888887984, 95.6608333333313]
{'GASEOSAS': 1, 'BEBIDAS DEPORTIVAS': 1, 'AGUA': 2, 'NECTAR': 2, 'CITRUS': 3, 'ENERGIZANTE': 8, 'DETERGENTE': 9, 'CAFE': 10, 'MASCOTA': 11, 'CONSERVA DE PESCADO': 12, 'DESINFECTANTE': 13, 'ACEITE': 14, 'SUERO': 15, 'GELATINA': 16, 'LAVAVAJILLAS': 17, 'SUAVIZANTES': 18, 'MOD

## 3.2. Filtro de SKU por Ruta
Cada Ruta tiene una cantidad de SKUs distintos, la recomendacion será distinta para cada ruta, sin embargo, aquellas rutas que tengan pocos SKUs serán tratadas como una sola ruta

In [60]:
rutas = (
    df_ventas.groupby(["cod_ruta"])["id_cliente"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()["cod_ruta"]
    .unique()
)

In [61]:
low_sku_ruta = []

**Para rutas que tengan más de 45 SKUS**

In [62]:
for ruta in rutas:
    print("*" * 21)
    print("Ruta:", ruta)
    temp = df_ventas[(df_ventas["cod_ruta"] == ruta)]
    print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
    # print(f"Giros en ruta {ruta}:")
    # print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
    if temp["cod_articulo_magic"].nunique() < 10:
        low_sku_ruta.append(ruta)
    else:
        temp.to_csv(f"Processed/rutas/D_{ruta}_ventas.csv", index=False)

*********************
Ruta: 3411
SKUs disponibles: 108
*********************
Ruta: 4217
SKUs disponibles: 110
*********************
Ruta: 5404
SKUs disponibles: 112
*********************
Ruta: 3304
SKUs disponibles: 119
*********************
Ruta: 4903
SKUs disponibles: 112
*********************
Ruta: 4504
SKUs disponibles: 109
*********************
Ruta: 2706
SKUs disponibles: 120
*********************
Ruta: 3303
SKUs disponibles: 126
*********************
Ruta: 5204
SKUs disponibles: 113
*********************
Ruta: 1309
SKUs disponibles: 109
*********************
Ruta: 2717
SKUs disponibles: 104
*********************
Ruta: 2904
SKUs disponibles: 113
*********************
Ruta: 2804
SKUs disponibles: 116
*********************
Ruta: 3403
SKUs disponibles: 114
*********************
Ruta: 3811
SKUs disponibles: 106
*********************
Ruta: 1217
SKUs disponibles: 99
*********************
Ruta: 1917
SKUs disponibles: 105
*********************
Ruta: 1904
SKUs disponibles: 119
***********

**Para rutas que tengan menos de 45 SKUS**

In [63]:
print("*" * 21)
print("Rutas:", low_sku_ruta)
temp = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
print("SKUs disponibles:", temp["cod_articulo_magic"].nunique())
# print(f"Giros en ruta {ruta}:")
# print(temp.groupby(["desc_giro"])["id_cliente"].nunique(dropna=False))
temp.to_csv("Processed/rutas/D_low_ruta_ventas.csv", index=False)

*********************
Rutas: []
SKUs disponibles: 0


# 4. Modelo Pedido Sugerido

## 4.1. Importando librerias necesarias

In [64]:
import os

from pyspark.sql import SparkSession, SQLContext
from pyspark.sql.types import StructType, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    when,
    col,
    regexp_replace,
    concat,
    countDistinct,
    lit,
    monotonically_increasing_id,
    hash,
)

from pyspark.ml.linalg import Vector
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [65]:
!java -version

openjdk version "11.0.30" 2026-01-20 LTS
OpenJDK Runtime Environment Corretto-11.0.30.7.1 (build 11.0.30+7-LTS)
OpenJDK 64-Bit Server VM Corretto-11.0.30.7.1 (build 11.0.30+7-LTS, mixed mode)


In [66]:
import os
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

## 4.2 Construccion del algoritmo ALS

In [67]:
def als_training_job(cedi, sku_len):

    spark = (
        SparkSession.builder.config("spark.memory.offHeap.enabled", "true")
        .config("spark.memory.offHeap.size", "10G")
        .appName("K-Means for ALS")
        .getOrCreate()
    )

    ventas = (
        spark.read.format("csv")
        .options(header=True, delimiter=",")
        .load(f"Processed/rutas/D_{cedi}_ventas.csv")
    )

    ventas = ventas.na.drop(subset=["fecha_liquidacion"])
    ventas = ventas.groupBy(
        ["id_cliente", "cod_articulo_magic", "cod_compania", "cod_cliente"]
    ).agg(countDistinct("fecha_liquidacion"))
    ventas = ventas.select(
        col("id_cliente").alias("id_cliente"),
        col("count(fecha_liquidacion)").alias("frecuencia"),
        col("cod_articulo_magic").alias("cod_articulo_magic"),
        col("cod_compania").alias("cod_compania"),
        col("cod_cliente").alias("cod_cliente"),
    )

    input_cols = ["frecuencia"]
    vec_assembler = VectorAssembler(inputCols=input_cols, outputCol="features")
    ventas_kmeans = vec_assembler.transform(ventas)
    ventas_kmeans = ventas_kmeans.fillna(0.0, subset=["frecuencia"])

    kmeans = KMeans(featuresCol="features", k=5)
    model = kmeans.fit(ventas_kmeans)

    resumen_ventas = model.transform(ventas_kmeans).groupby("prediction").max()
    resumen_ventas.orderBy(["max(frecuencia)"], ascending=0)
    resumen_ventas = resumen_ventas.select(
        col("prediction").alias("prediction"), col("max(frecuencia)").alias("max")
    )

    maxes = resumen_ventas.select("max").toPandas()["max"]
    maxes = maxes.astype(float)
    maxes = list(maxes)

    labels = [5, 4, 3, 2, 1]
    rank_map = dict(zip(labels, maxes))

    ventas = ventas.withColumn("frecuencia", col("frecuencia").cast(DoubleType()))
    ventas = ventas.withColumn(
        "rating",
        when(ventas["frecuencia"] < rank_map.get(labels[-1]), labels[-1])
        .when(ventas["frecuencia"] < rank_map.get(labels[-2]), labels[-2])
        .when(ventas["frecuencia"] < rank_map.get(labels[-3]), labels[-3])
        .when(ventas["frecuencia"] < rank_map.get(labels[-4]), labels[-4])
        .otherwise(labels[-5]),
    )

    cols = ["id_cliente", "cod_compania", "cod_cliente", "cod_articulo_magic", "rating"]
    als_records = ventas.select(*cols)

    # als_records = als_records.withColumn("clienteId",concat("cod_compania","cod_cliente"))
    als_records = als_records.withColumn(
        "clienteId", concat(col("cod_compania"), lit("|"), col("cod_cliente"))
    )
    # als_records = als_records.withColumn("clienteId", col("clienteId").cast(IntegerType()))
    # Crear una columna numérica para usar en el modelo ALS
    # als_records = als_records.withColumn("clienteId_numeric", monotonically_increasing_id())
    als_records = als_records.withColumn("clienteId_numeric", hash(col("clienteId")))
    als_records = als_records.withColumn(
        "cod_articulo_magic", col("cod_articulo_magic").cast(IntegerType())
    )

    als_records = als_records.na.drop(
        subset=["clienteId_numeric", "cod_articulo_magic"]
    )

    input_cols = ["clienteId", "clienteId_numeric", "cod_articulo_magic", "rating"]
    als_records = als_records.select(*input_cols)

    als_records = als_records.dropDuplicates(["clienteId", "cod_articulo_magic"])

    als = ALS(
        rank=10,
        maxIter=5,
        implicitPrefs=True,
        ratingCol="rating",
        itemCol="cod_articulo_magic",
        userCol="clienteId_numeric",
    )
    model = als.fit(als_records)

    # obtener matrices de descomposicion (usuario y articulos)
    user_matrix = model.userFactors
    item_matrix = model.itemFactors

    user_matrix_pd = user_matrix.toPandas()
    item_matrix_pd = item_matrix.toPandas()

    recs = model.recommendForAllUsers(sku_len)
    # recs = recs.select("ClienteId","recommendations.cod_articulo_magic")
    # recs_to_parse = recs.toPandas()

    recs = recs.select("clienteId_numeric", "recommendations.cod_articulo_magic")
    # Hacer un join con als_records para recuperar la columna 'clienteId' original
    recs = recs.join(
        als_records.select("clienteId", "clienteId_numeric"),
        on="clienteId_numeric",
        how="left",
    )

    # Convertir a pandas
    recs_to_parse = recs.select("clienteId", "cod_articulo_magic").toPandas()

    lista_rec = [f"r{i+1}" for i in range(sku_len)]
    # recs_to_parse[lista_rec] = pd.DataFrame(recs_to_parse['cod_articulo_magic'].tolist(), index= recs_to_parse.index)
    new_cols = pd.DataFrame(
        recs_to_parse["cod_articulo_magic"].tolist(),
        index=recs_to_parse.index,
        columns=lista_rec,
    )
    recs_to_parse = pd.concat([recs_to_parse, new_cols], axis=1)
    client_recs = pd.melt(
        recs_to_parse, id_vars=["clienteId"], value_vars=lista_rec
    )

    # client_recs['compania'] = client_recs["clienteId"].apply(lambda x: str(x)[:2])
    client_recs["compania"] = client_recs["clienteId"].str.split("|").str[-2]
    # client_recs['cliente'] = client_recs["clienteId"].apply(lambda x: str(x)[2:])
    #client_recs["cliente"] = client_recs["clienteId"].str.split("|").str[1].apply(lambda x: str(x)[:])
    client_recs["cliente"] = client_recs["clienteId"].str.split("|").str[-1]
    client_recs["compania"] = client_recs["compania"].apply(lambda x: x.rjust(4, "0"))
    client_recs["id_cliente"] = client_recs["compania"] + "|" + client_recs["cliente"]
    client_recs["cod_articulo_magic"] = client_recs["value"]
    
    # print(client_recs)

    client_recs = (
        client_recs[["id_cliente", "cod_articulo_magic"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Obtener el nombre del archivo CSV
    nombre_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo ya existe
    try:
        # Intentar abrir el archivo para lectura
        with open(nombre_archivo, "r") as f:
            # Si el archivo existe, header=False para no escribir cabeceras
            client_recs.to_csv(
                nombre_archivo, mode="a", header=False, index=False, sep=";"
            )
    except FileNotFoundError:
        # Si el archivo no existe, header=True para escribir las cabeceras
        client_recs.to_csv(nombre_archivo, mode="a", header=True, index=False, sep=";")

## 4.3 Corriendo PS

In [68]:
def runALS():
    # Ruta del archivo que deseas verificar y eliminar si existe
    ruta_archivo = "Output/D_rutas_rec.csv"
    # Verificar si el archivo existe
    if os.path.exists(ruta_archivo):
        # Eliminar el archivo si existe
        os.remove(ruta_archivo)
        print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
    else:
        print(f"El archivo en la ruta {ruta_archivo} no existe.")
    # CORRIENDO ALS PARA RUTAS DE MAS DE 45 SKUs
    for ruta in rutas:
        print("*" * 21)
        temp = pd.read_csv(f"Processed/rutas/D_{ruta}_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en ruta {ruta}:", sku_len)
        als_training_job(ruta, sku_len)
    # CORRIENDO ALS PARA rutas que tengan menos de 45 SKUS
    if len(low_sku_ruta) != 0:
        temp = pd.read_csv(f"Processed/rutas/D_low_ruta_ventas.csv")
        sku_len = temp["cod_articulo_magic"].nunique()
        print(f"SKUs disponibles en low ruta {low_sku_ruta}:", sku_len)
        als_training_job("low_ruta", sku_len)
    else:
        print("No existen rutas con pocos skus :D")

In [69]:
rutas = rutas[~pd.Series(rutas).isin(pd.Series(low_sku_ruta))]
rutas

array([3411, 4217, 5404, 3304, 4903, 4504, 2706, 3303, 5204, 1309, 2717,
       2904, 2804, 3403, 3811, 1217, 1917, 1904, 4517, 4604, 3504, 4711,
       5703, 2011, 3211, 1417, 2107, 4704, 2809, 4804, 1211, 2307, 2411,
       5104, 2504, 2209, 1911, 1706, 2106, 2111, 3117, 1704, 3611, 4904,
       1403, 1104, 4617, 1108, 3104, 2911, 3517, 2008, 3011, 3204, 1517,
       5603, 2404, 2304, 2417, 3806, 1607, 2311, 2406, 1906, 2204, 2611,
       2704, 3111, 3004, 2009, 1504, 1308, 1011, 1609, 1804, 2306, 3911,
       1809, 2709, 1408, 2211, 2604, 1304, 1204, 3404, 2517, 3511, 1409,
       2711, 2407, 1808, 1608, 1806, 4017, 1604, 2109, 1807, 2207, 2811,
       3106, 2208, 2006, 3217, 4917, 1506, 1907, 3311, 1306, 1208, 1511,
       2206, 1206, 1404, 3917, 1717, 3507, 2108, 1103, 1611, 1711, 4811,
       2004, 2104, 2003, 1311, 2511, 1209, 2617, 1411, 2807, 2817, 4117,
       2703, 1009, 1008, 1004, 1909, 1709, 1908, 2917, 2303, 2806, 4317,
       4011, 2606, 2403, 2607, 1606, 1006, 1817, 36

In [70]:
%%time
runALS()

El archivo en la ruta Output/D_rutas_rec.csv ha sido eliminado.
*********************
SKUs disponibles en ruta 3411: 108


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/30 20:30:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 20:31:04 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/30 20:31:04 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
26/03/30 20:31:05 WARN InstanceBuilder$NativeLAPACK: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


*********************
SKUs disponibles en ruta 4217: 110
*********************
SKUs disponibles en ruta 5404: 112
*********************
SKUs disponibles en ruta 3304: 119
*********************
SKUs disponibles en ruta 4903: 112
*********************
SKUs disponibles en ruta 4504: 109
*********************
SKUs disponibles en ruta 2706: 120
*********************
SKUs disponibles en ruta 3303: 126
*********************
SKUs disponibles en ruta 5204: 113
*********************
SKUs disponibles en ruta 1309: 109
*********************
SKUs disponibles en ruta 2717: 104
*********************
SKUs disponibles en ruta 2904: 113
*********************
SKUs disponibles en ruta 2804: 116
*********************
SKUs disponibles en ruta 3403: 114
*********************
SKUs disponibles en ruta 3811: 106
*********************
SKUs disponibles en ruta 1217: 99
*********************
SKUs disponibles en ruta 1917: 105
*********************
SKUs disponibles en ruta 1904: 119
*********************
SKUs dispo

# 5. Ajustes al output del algoritmo ALS

## 5.-9 Quedarnos con skus que hayan tenido ventas en los ultimos 30 dias

In [71]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [72]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [73]:
pan_rec

,id_cliente,cod_articulo_magic,cod_ruta
0,EC|0090|478165,523104,3411
1,EC|0090|48450,508586,3411
2,EC|0090|240506,523108,3411
3,EC|0090|483904,511920,3411
4,EC|0090|67099,523079,3411
...,...,...,...
2026381,EC|0090|484405,800074,3107
2026382,EC|0090|484917,514445,3107
2026383,EC|0090|483898,800075,3107
2026384,EC|0090|484405,800075,3107


In [74]:
pan_rec.id_cliente.nunique()

18578

In [75]:
%%time
df_ventas = pd.read_parquet("Processed/ecuador_ventas_manana.parquet")

CPU times: user 4.93 s, sys: 1.82 s, total: 6.76 s
Wall time: 2.52 s


In [76]:
df_ventas["fecha_liquidacion"] = pd.to_datetime(df_ventas["fecha_liquidacion"])

In [77]:
# Definir el umbral de tiempo (últimos 30 días desde hoy)
# bajamos de 30 a 14 dias 2025-07-25
hoy = pd.Timestamp.today()
fecha_limite = hoy - pd.Timedelta(days=14)
print(fecha_limite)
# Filtrar ventas en los últimos 30 días
ventas_filtradas = df_ventas[df_ventas["fecha_liquidacion"] >= fecha_limite]

2026-03-16 20:47:22.221125


In [78]:
# Obtener los productos vendidos en cada ruta
productos_por_ruta = ventas_filtradas.groupby("cod_ruta")["cod_articulo_magic"].unique().reset_index()

In [79]:
# Unir con los productos vendidos en cada ruta
recomendaciones_validas = pan_rec.merge(productos_por_ruta, on="cod_ruta", how="inner")

In [80]:
recomendaciones_validas.columns = ['id_cliente', 'cod_articulo_magic', 'cod_ruta','lista_sku_ruta']

In [81]:
recomendaciones_validas.shape

(2026386, 4)

In [82]:
recomendaciones_validas  = recomendaciones_validas[recomendaciones_validas.apply(lambda row: row["cod_articulo_magic"] in row["lista_sku_ruta"], axis=1)].reset_index(drop=True)

In [83]:
recomendaciones_validas.shape

(1106129, 4)

In [84]:
recomendaciones_validas.head()

,id_cliente,cod_articulo_magic,cod_ruta,lista_sku_ruta
0,EC|0090|478165,523104,3411,"[509545, 508586, 524188, 523075, 524189, 51726..."
1,EC|0090|48450,508586,3411,"[509545, 508586, 524188, 523075, 524189, 51726..."
2,EC|0090|240506,523108,3411,"[509545, 508586, 524188, 523075, 524189, 51726..."
3,EC|0090|483904,511920,3411,"[509545, 508586, 524188, 523075, 524189, 51726..."
4,EC|0090|67099,523079,3411,"[509545, 508586, 524188, 523075, 524189, 51726..."


In [85]:
pan_rec = recomendaciones_validas[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [86]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,EC|0090|478165,523104
1,EC|0090|48450,508586
2,EC|0090|240506,523108
3,EC|0090|483904,511920
4,EC|0090|67099,523079


In [87]:
pan_rec.shape

(1106129, 2)

In [88]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-8 Recomendar de acuerdo a Subida, Bajada, Mantener

In [89]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [90]:
pan_rec.shape

(1106129, 2)

In [91]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [92]:
# df_ventas = pd.read_csv("Processed/ecuador_ventas_manana.csv")

In [93]:
def desplazar_nulos_fila(fila):
    return pd.Series(sorted(fila, key=pd.isna), index=fila.index)

In [94]:
# Definir una función para asignar los valores subida, mantener y bajar
def clasificar_valor(x):
    if x > 0:
        return "S"
    elif x == 0:
        return "M"
    else:
        return "B"

In [95]:
# df_ventas['fecha_liquidacion'] = pd.to_datetime(df_ventas['fecha_liquidacion'])

# Definir la fecha de referencia (hoy)
fecha_actual = datetime.now(pytz.timezone("America/Lima"))#.strftime('%Y-%m-%d')
fecha_30dias_atras = (fecha_actual - pd.Timedelta(days=30)).strftime('%Y-%m-%d')
fecha_60dias_atras = (fecha_actual - pd.Timedelta(days=60)).strftime('%Y-%m-%d')

print("fecha_actual",fecha_actual)
print("fecha_30dias_atras",fecha_30dias_atras)
print("fecha_60dias_atras",fecha_60dias_atras)

fecha_actual 2026-03-30 15:47:46.860127-05:00
fecha_30dias_atras 2026-02-28
fecha_60dias_atras 2026-01-29


In [96]:
# Filtrar los últimos 30 días
df_ultimos_30 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_30dias_atras) 
                          & (df_ventas['fecha_liquidacion'] <= fecha_actual.strftime('%Y-%m-%d'))]

# Filtrar los días 31 a 60
df_31_60 = df_ventas[(df_ventas['fecha_liquidacion'] > fecha_60dias_atras)
              & (df_ventas['fecha_liquidacion'] <= fecha_30dias_atras)]

# Agrupar por codigo_usuario, ruta, codigo_producto y sumar ventas
ventas_ultimos_30 = df_ultimos_30.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_ultimos_30["mes"] = "0_30"
ventas_31_60 = df_31_60.groupby(['cod_ruta', 'cod_articulo_magic'])['imp_netovta'].sum().reset_index()
ventas_31_60["mes"] = "31_60"

df_grouped = pd.concat([ventas_ultimos_30,ventas_31_60],axis=0,ignore_index=True)
df_grouped = pd.pivot_table(
    df_grouped,
    values="imp_netovta",
    index=["cod_ruta", "cod_articulo_magic"],
    columns=["mes"],
    aggfunc="sum",
).reset_index()

# Rellenando con 0 si es nulo
df_grouped["0_30"] = df_grouped["0_30"].fillna(0)
df_grouped["31_60"] = df_grouped["31_60"].fillna(0)

# Asignando variacion de productos por mes
df_grouped["v1_2"] = (
    (df_grouped[df_grouped.columns[2]] - df_grouped[df_grouped.columns[3]])
    / df_grouped[df_grouped.columns[3]]
    * 100
)
df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)
df_grouped["v1_2"] = df_grouped["v1_2"].fillna(-1)
df_grouped["vp"] = (df_grouped["v1_2"])

/tmp/ipykernel_31118/261865676.py:34: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_grouped["v1_2"].replace([float('inf'), float('-inf')], -1, inplace=True)


In [97]:
# Aplicar la función a la columna 'valor' y guardar el resultado en una nueva columna 'flag'
df_grouped["flag"] = df_grouped["vp"].apply(lambda x: clasificar_valor(x))
# Definir el mapeo de valores
mapeo_flag = {"S": 0, "M": 1, "B": 2}
# Aplicar el mapeo a la columna 'clasificacion' y guardar el resultado en una nueva columna 'flag_rank'
df_grouped["flag_rank"] = df_grouped["flag"].map(mapeo_flag)

In [98]:
df_grouped

mes,cod_ruta,cod_articulo_magic,0_30,31_60,v1_2,vp,flag,flag_rank
0,1003,500010,143.4881,218.4013,-34.300712,-34.300712,B,2
1,1003,500147,473.7926,595.7294,-20.468488,-20.468488,B,2
2,1003,500187,5.8472,5.8472,0.000000,0.000000,M,1
3,1003,508462,284.0023,280.9151,1.098980,1.098980,S,0
4,1003,508514,881.5302,550.9563,60.000022,60.000022,S,0
...,...,...,...,...,...,...,...,...
16851,5717,624067,0.0000,0.8006,-100.000000,-100.000000,B,2
16852,5717,699814,0.0000,1.3050,-100.000000,-100.000000,B,2
16853,5717,800072,0.0000,10.0500,-100.000000,-100.000000,B,2
16854,5717,800073,3.3500,13.4000,-75.000000,-75.000000,B,2


In [99]:
pan_rec.shape

(1106129, 3)

In [100]:
pan_rec = pd.merge(pan_rec,df_grouped[["cod_ruta", "cod_articulo_magic","flag_rank"]],on=["cod_ruta", "cod_articulo_magic"],how="left")

In [101]:
pan_rec["flag_rank"] = pan_rec["flag_rank"].fillna(3)

In [102]:
# Agregar una columna con el orden original
pan_rec["original_order"] = pan_rec.index

In [103]:
# Ordenar por cod_cliente, rank_producto (ascendente) y mantener el orden original en caso de empate
pan_rec = pan_rec.sort_values(by=["id_cliente", "flag_rank", "original_order"], ascending=[True, True, True])#.drop(columns=["original_order"])

In [104]:
pan_rec.head()

,id_cliente,cod_articulo_magic,cod_ruta,flag_rank,original_order
619598,EC|0090|10003,515519,4917,0,619598
619941,EC|0090|10003,515417,4917,0,619941
620027,EC|0090|10003,515408,4917,0,620027
620111,EC|0090|10003,508514,4917,0,620111
620187,EC|0090|10003,599211,4917,0,620187


In [105]:
pan_rec.shape

(1106129, 5)

In [106]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]].reset_index(drop=True)

In [107]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,EC|0090|10003,515519
1,EC|0090|10003,515417
2,EC|0090|10003,515408
3,EC|0090|10003,508514
4,EC|0090|10003,599211


In [108]:
pan_rec.shape

(1106129, 2)

In [109]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-7 Usamos el archivo de validacion para no recomendar SKUs que no se deben a las rutas

In [110]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [111]:
pan_rec.shape

(1106129, 2)

In [112]:
pan_rec = pd.merge(pan_rec,pan_ventas[["id_cliente","cod_ruta"]].drop_duplicates(),on="id_cliente",how="left")

In [113]:
pan_rec.isnull().sum()

id_cliente            0
cod_articulo_magic    0
cod_ruta              0
dtype: int64

In [114]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
1003    70
1004    60
1006    52
1007    66
1008    52
Name: cod_articulo_magic, dtype: int64

In [115]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

58.75925925925926

In [116]:
# Ruta del archivo en S3
s3_path = "s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/ecuador/maestro_productos_ecuador000"

# Leer el CSV directamente a un DataFrame
skus_val = pd.read_csv(s3_path,sep = ";")
# skus_val = pd.read_csv("maestra_productos_ecuador_20250217.csv",sep=";")
# skus_val.head()

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [117]:
skus_val = skus_val[skus_val.cod_compania==90]
# Nos quedamos solo con las filas del proceso de hoy
# Obtener la fecha de hoy en el mismo formato
hoy = int(datetime.now(pytz.timezone("America/Lima")).strftime('%Y%m%d'))
print("fecha_proceso: ", hoy)
# Filtrar solo las filas con la fecha de hoy
# skus_val = skus_val[skus_val['fecha_proceso'] == hoy]
skus_val["cod_compania"] = (skus_val["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
skus_val["id_cliente"] = (
    "EC"
    + "|"
    + skus_val["cod_compania"].astype(str)
    + "|"
    + skus_val["cod_cliente"].astype(str)
)

fecha_proceso:  20260330


In [118]:
skus_val.head()

,cod_compania,desc_compania,cod_sucursal,desc_sucursal,cod_zona,desc_zona,cod_ruta,desc_ruta,cod_modulo,desc_modulo,cod_cliente,cod_articulo_magic,desc_articulo,id_cliente
0,0090,AJECUADOR S.A.,17,CEDIS GUAYAQUIL SUR,90117,90117 - TRADICIONAL,1517,TRADICIONAL,145,Ma-Vi,365059,599653,BIG COLA PET NO RETORNABLE 1100 ML 12,EC|0090|365059
1,0090,AJECUADOR S.A.,9,CEDIS SANTO DOMINGO,90209,TRADICIONAL,1109,TRADICIONAL,592,Ma-Vi,409431,500147,BIG COLA PET NO RETORNABLE 2250 ml 6 pack,EC|0090|409431
2,0090,AJECUADOR S.A.,4,CEDIS QUITO NORTE,90104,90104 - TRADICIONAL,1004,1004 - TRADICIONAL,405,Lu-Ju,453592,516810,SPORADE BLUEBERRY REGULAR PET NO RETORNABLE 12...,EC|0090|453592
3,0090,AJECUADOR S.A.,11,CEDIS QUITO SUR,90111,TRADICIONAL,1211,TRADICIONAL,513,Ma-Vi,487457,509728,CIFRUT TROPICAL PUNCH PET NO RETORNABLE 1000 M...,EC|0090|487457
4,0090,AJECUADOR S.A.,17,CEDIS GUAYAQUIL SUR,90217,90217 - TRADICIONAL,2417,TRADICIONAL,175,Mi-Sa,429253,508586,CIELO AGUA PET NO RETORNABLE 1200 ML 12,EC|0090|429253


In [119]:
skus_val.shape

(1787798, 14)

In [120]:
# skus_val = pd.read_csv("productos_pedido_sugerido_validacion_cam_090225.csv",sep=";")
# skus_val = skus_val[skus_val.cod_compania==70].reset_index(drop=True)

In [121]:
# Cada ruta pertenece a una compañia
skus_val.groupby("cod_ruta").cod_compania.nunique().mean()

1.0

In [122]:
pan_rec = pd.merge(pan_rec,skus_val[['cod_articulo_magic', 'id_cliente']].drop_duplicates(),left_on=["id_cliente","cod_articulo_magic"],right_on=["id_cliente","cod_articulo_magic"],how="inner")

In [123]:
pan_rec.shape

(497912, 3)

In [124]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().head()

cod_ruta
1003    70
1004    60
1006    52
1007    66
1008    52
Name: cod_articulo_magic, dtype: int64

In [125]:
pan_rec.groupby("cod_ruta").cod_articulo_magic.nunique().mean()

58.75925925925926

In [126]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

26.812708669897685

In [127]:
pan_rec.id_cliente.nunique()

18570

In [128]:
pan_rec = pan_rec[["id_cliente","cod_articulo_magic"]]

In [129]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,EC|0090|10003,515519
1,EC|0090|10003,515417
2,EC|0090|10003,515408
3,EC|0090|10003,508514
4,EC|0090|10003,599211


In [130]:
pan_rec.shape

(497912, 2)

In [131]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-5 Cruzar recomendaciones con df_quiebres (STOCK)

## 5.-5 Filtrar SKUs por STOCK

In [132]:
stock = pd.read_csv("s3://aje-prd-analytics-artifacts-s3/pedido_sugerido/data-v1/ecuador/D_stock_ec.csv")

In [133]:
stock.drop(columns=["Fecha","Database"], inplace=True)
stock.columns=["cod_compania","cod_sucursal","cod_articulo_magic","stock_cf"]

In [134]:
stock["cod_compania"] = (stock["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
stock["cod_sucursal"] = (stock["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [135]:
fecha_hace_12_dias = (datetime.today() - timedelta(days=12)).strftime('%Y-%m-%d')
print(fecha_hace_12_dias)

2026-03-18


In [136]:
# Promedio diario de cajas fisicas por sucursal y compañia de los ultimos 12 dias
prom_diario_vta = df_ventas[df_ventas.cant_cajafisicavta > 0]
prom_diario_vta = df_ventas[df_ventas.fecha_liquidacion>=fecha_hace_12_dias].groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic","fecha_liquidacion"]).cant_cajafisicavta.sum().reset_index().groupby(
    ["cod_compania","cod_sucursal","cod_articulo_magic"]).cant_cajafisicavta.mean().reset_index()
prom_diario_vta["cod_compania"] = (prom_diario_vta["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
prom_diario_vta["cod_sucursal"] = (prom_diario_vta["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [137]:
df_stock = pd.merge(prom_diario_vta,stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"],how="left")
df_stock["dias_stock"] = df_stock["stock_cf"]/df_stock["cant_cajafisicavta"]
df_stock =  df_stock[(df_stock.dias_stock > 3) & (df_stock.cant_cajafisicavta > 0)].reset_index(drop=True)

In [138]:
pan_rec.nunique()

id_cliente            18570
cod_articulo_magic      124
dtype: int64

In [139]:
pan_rec.groupby("id_cliente").cod_articulo_magic.nunique().mean()

26.812708669897685

In [140]:
pan_rec.shape

(497912, 2)

In [141]:
# Cruzamos stock

In [142]:
test = pd.merge(pan_rec,df_ventas[["id_cliente","cod_compania","cod_sucursal"]].drop_duplicates(),on="id_cliente",how="left")

In [143]:
test["cod_compania"] = (test["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0")))
test["cod_sucursal"] = (test["cod_sucursal"].astype(str).apply(lambda x: str(int(x)).rjust(2, "0")))

In [144]:
test_f = pd.merge(test, df_stock, on=["cod_compania","cod_sucursal","cod_articulo_magic"], how="inner")

In [145]:
test_f.head()

,id_cliente,cod_articulo_magic,cod_compania,cod_sucursal,cant_cajafisicavta,stock_cf,dias_stock
0,EC|0090|10003,515519,0090,17,267.071429,3699,13.850227
1,EC|0090|10003,515417,0090,17,218.500000,1227,5.615561
2,EC|0090|10003,515408,0090,17,269.770833,1352,5.011661
3,EC|0090|10003,508514,0090,17,685.000000,2297,3.353285
4,EC|0090|10003,599211,0090,17,22.330357,236,10.568573


In [146]:
test_f.shape

(419733, 7)

In [147]:
test_f.nunique()

id_cliente            18560
cod_articulo_magic      120
cod_compania              1
cod_sucursal              8
cant_cajafisicavta      545
stock_cf                482
dias_stock              671
dtype: int64

In [148]:
PS_VAL_promedio_recs_por_cliente_stock = (
    test_f.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()
)
PS_VAL_promedio_recs_por_cliente_stock

22.61325431034483

In [149]:
pan_rec = test_f[["id_cliente","cod_articulo_magic"]].copy()

In [150]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-4 Mantener Solo SKUS en especifico (solo usar si se tiene el excel con SKU activos -> el input de ventas ya deberia incluir este filtro)

In [151]:
# pan_rec=pd.read_csv("Output/D_rutas_rec.csv", sep=";")
# pan_rec["id_cliente"] = "EC|" + pan_rec["id_cliente"]

In [152]:
# pan_rec.groupby("id_cliente")["cod_articulo_magic"].nunique().mean()

In [153]:
# pan_rec.groupby("id_cliente")["cod_articulo_magic"].count().mean()

In [154]:
# sku_con_precio=pd.read_excel("Input/LISTA DE PRECIOS ECUADOR_2024-11-08.xlsx",sheet_name="Hoja1")

In [155]:
# # Original size de recomendacioness
# pan_rec.shape

In [156]:
# pan_rec = pan_rec[pan_rec.cod_articulo_magic.isin(sku_con_precio["SKU"].unique())].reset_index(drop=True)

In [157]:
# # Size de recomendaciones luego de filtar solo a los del excel de SKUs activos
# pan_rec.shape

In [158]:
# pan_rec.groupby("id_cliente")["cod_articulo_magic"].count().mean()

In [159]:
# pan_rec["cod_articulo_magic"].nunique()

In [160]:
# pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";",index=False)

## 5.-3 Quitar Recomendaciones de SKUS en especifico

In [161]:
pan_rec=pd.read_csv("Output/D_rutas_rec.csv", sep=";")
#pan_rec["id_cliente"]="EC|"+pan_rec["id_cliente"]

In [162]:
skus_sin_precio=[508585,516617,514445,514503,515907,516810,509727,511922,599890]

In [163]:
pan_rec.shape

(419733, 2)

In [164]:
pan_rec=pan_rec[~(pan_rec["cod_articulo_magic"].isin(skus_sin_precio))].reset_index(drop=True)

In [165]:
pan_rec.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.-2 Quitar Recomendaciones de los ultimos 14 dias

In [166]:
pan_rec=pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [167]:
master_prod=pd.read_csv("Input/EC_maestro_productos.csv")
master_prod

,cod_articulo_magic,desc_articulo
0,60968,PREFORMA DE 17.15 GRS CRISTAL AGUA PT
1,60969,PREFORMA DE 17.15 GRS CRISTAL CON UV PT
2,60970,PREFORMA DE 17.15 GRS CRISTAL PT
3,60971,PREFORMA DE 17.15 GRS AZUL PT
4,60972,PREFORMA DE 17.15 GRS AZUL AGUA PT
...,...,...
7951,524187,CIELO MANZANA PET NO RETORNABLE 600 ML 6
7952,624212,CIFRUT NARANJILLA PET NO RETORNABLE 1700 ML 6 MC
7953,524212,CIFRUT NARANJILLA PET NO RETORNABLE 1700 ML 6
7954,523617,DILYTE FRESA KIWI PET NO RETORNABLE 650 ML 6


In [168]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [169]:
ruta_archivo = f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv"
# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    # Eliminar el archivo si existe
    os.remove(ruta_archivo)
    print(f"El archivo en la ruta {ruta_archivo} ha sido eliminado.")
else:
    print(f"El archivo en la ruta {ruta_archivo} no existe.")

El archivo en la ruta Output/PS_piloto_v1/D_base_pedidos_2026-03-31.csv no existe.


In [170]:
# Directorio donde se encuentran los archivos
directorio = 'Output/PS_piloto_v1/'

# Expresión regular para encontrar fechas en el nombre de los archivos
patron_fecha = r'\d{4}-\d{2}-\d{2}'

# Lista para almacenar las fechas extraídas
fechas = []

# Iterar sobre los archivos en el directorio
for archivo in os.listdir(directorio):
    # Verificar si el archivo es un archivo CSV y coincide con el patrón de nombres
    if archivo.endswith('.csv') and re.match(r'^D_base_pedidos_\d{4}-\d{2}-\d{2}\.csv$', archivo):
        # Extraer la fecha del nombre del archivo
        fecha = re.search(patron_fecha, archivo).group()
        # Agregar la fecha a la lista
        fechas.append(fecha)

In [171]:
last_7_days_2=sorted(fechas)[-14:]
#last_7_days_2=last_7_days_2[:-1]
last_7_days_2

['2026-03-14',
 '2026-03-16',
 '2026-03-17',
 '2026-03-18',
 '2026-03-19',
 '2026-03-20',
 '2026-03-21',
 '2026-03-23',
 '2026-03-24',
 '2026-03-25',
 '2026-03-26',
 '2026-03-27',
 '2026-03-28',
 '2026-03-30']

In [172]:
#Leer recomendaciones de los ultimos 7 dias y cruzar con clientes que tienen recomendacion para mañana
last_7_days_recs=pd.DataFrame()
for fecha_rec in last_7_days_2:
    df_temp=pd.read_csv(f"Output/PS_piloto_v1/D_base_pedidos_{fecha_rec}.csv",dtype={"Compania":"str","Cliente":"str"})
    df_temp["id_cliente"]='EC|'+df_temp['Compania']+'|'+df_temp['Cliente']
    df_temp=df_temp[df_temp["id_cliente"].isin(pan_rec["id_cliente"].unique())]
    last_7_days_recs=pd.concat([last_7_days_recs,df_temp],axis=0)
    print(f"{fecha_rec} done")

2026-03-14 done
2026-03-16 done
2026-03-17 done
2026-03-18 done
2026-03-19 done
2026-03-20 done
2026-03-21 done
2026-03-23 done
2026-03-24 done
2026-03-25 done
2026-03-26 done
2026-03-27 done
2026-03-28 done
2026-03-30 done


In [173]:
# Utilizamos merge para combinar los dataframes, utilizando la columna 'Cliente' como clave
df_combinado = pd.merge(pan_rec, last_7_days_recs, left_on=['id_cliente', 'cod_articulo_magic'],right_on=['id_cliente', 'Producto'], how='left', indicator=True)
# Filtramos los registros que solo están en el DataFrame 1
df_resultado = df_combinado[df_combinado['_merge'] == 'left_only'][["id_cliente","cod_articulo_magic"]].reset_index(drop=True)
df_resultado

,id_cliente,cod_articulo_magic
0,EC|0090|10003,515408
1,EC|0090|10003,508514
2,EC|0090|10003,599211
3,EC|0090|10003,515912
4,EC|0090|10003,524187
...,...,...
267437,EC|0090|9999,514411
267438,EC|0090|9999,621870
267439,EC|0090|9999,623423
267440,EC|0090|9999,523106


In [174]:
df_resultado.shape

(267442, 2)

In [175]:
df_resultado.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

##  Quitar Recurrente y Estrategico

In [176]:
fecha_tomorrow

'2026-03-31'

In [177]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [178]:
pan_rec.head()

,id_cliente,cod_articulo_magic
0,EC|0090|10003,515408
1,EC|0090|10003,508514
2,EC|0090|10003,599211
3,EC|0090|10003,515912
4,EC|0090|10003,524187


In [179]:
# pe = pd.read_csv(f"s3://aje-analytics-ps-backup/Pedido_Estrategico/Ecuador/Output/estr_base_pedidos_{fecha_tomorrow}.csv")
pr = pd.read_csv(f"s3://aje-analytics-ps-backup/Pedido_Recurrente/Ecuador/Output/recu_base_pedidos_{fecha_tomorrow}.csv")

In [180]:
# pe.head()

In [181]:
pr.head()

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,tipoRecomendacion,ultFecha,Destacar
0,EC,90,3,109330,96,508514,1,0,2026-03-31,PR1,2026-02-28,True
1,EC,90,3,109330,96,515408,1,0,2026-03-31,PR2,2026-03-11,True
2,EC,90,3,109330,96,514422,1,0,2026-03-31,PR3,2026-03-18,True
3,EC,90,3,109330,96,515904,1,0,2026-03-31,PR4,2026-03-11,True
4,EC,90,3,109330,96,517260,1,0,2026-03-31,PR5,2026-03-11,True


In [182]:
# quitar_temp = pd.concat([pe, pr], ignore_index=True).drop_duplicates()
quitar_temp = pr.copy()

In [183]:
quitar_temp["Compania"] = quitar_temp["Compania"].astype(str).str.zfill(4)
quitar_temp["id_cliente"] = "EC" + '|' + quitar_temp['Compania'] + '|' + quitar_temp['Cliente'].astype(str)
quitar_temp.rename(columns={"Producto":"cod_articulo_magic"},inplace=True)

In [184]:
quitar_temp

,Pais,Compania,Sucursal,Cliente,Modulo,cod_articulo_magic,Cajas,Unidades,Fecha,tipoRecomendacion,ultFecha,Destacar,id_cliente
0,EC,0090,3,109330,96,508514,1,0,2026-03-31,PR1,2026-02-28,True,EC|0090|109330
1,EC,0090,3,109330,96,515408,1,0,2026-03-31,PR2,2026-03-11,True,EC|0090|109330
2,EC,0090,3,109330,96,514422,1,0,2026-03-31,PR3,2026-03-18,True,EC|0090|109330
3,EC,0090,3,109330,96,515904,1,0,2026-03-31,PR4,2026-03-11,True,EC|0090|109330
4,EC,0090,3,109330,96,517260,1,0,2026-03-31,PR5,2026-03-11,True,EC|0090|109330
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9072,EC,0090,3,9766,96,599653,1,0,2026-03-31,PR6,2026-03-14,True,EC|0090|9766
9073,EC,0090,3,9766,96,524212,1,0,2026-03-31,PR7,2026-03-07,True,EC|0090|9766
9074,EC,0090,3,9766,96,599479,1,0,2026-03-31,PR8,2026-03-14,True,EC|0090|9766
9075,EC,0090,3,9766,96,514422,1,0,2026-03-31,PR9,2026-03-28,True,EC|0090|9766


In [185]:
pan_rec.shape

(267442, 2)

In [186]:
# quitamos estrategico y recurrente a nivel cliente
 
# 1. Hacemos un merge por las columnas de coincidencia usando indicator=True
merge_temp = pan_rec.merge(
    quitar_temp[['id_cliente', 'cod_articulo_magic']], 
    on=['id_cliente', 'cod_articulo_magic'], 
    how='left', 
    indicator=True
)

# 2. Nos quedamos solo con las filas que solo existen en pan_rec (left_only)
rec_sin_pe_ni_pr = merge_temp[merge_temp['_merge'] == 'left_only'].drop(columns=['_merge'])

In [187]:
rec_sin_pe_ni_pr.shape

(261770, 2)

In [188]:
rec_sin_pe_ni_pr

,id_cliente,cod_articulo_magic
0,EC|0090|10003,515408
1,EC|0090|10003,508514
2,EC|0090|10003,599211
3,EC|0090|10003,515912
4,EC|0090|10003,524187
...,...,...
267437,EC|0090|9999,514411
267438,EC|0090|9999,621870
267439,EC|0090|9999,623423
267440,EC|0090|9999,523106


In [189]:
rec_sin_pe_ni_pr.to_csv("Output/D_rutas_rec.csv", sep=";", index=False)

## 5.0 Calcular Irregularidad de clientes

In [190]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [191]:
pan_rec["id_cliente"].nunique()

18381

In [192]:
now = pd.to_datetime(datetime.now(pytz.timezone("America/Lima")).strftime("%Y-%m-01"))
fecha_doce_meses_atras = now - pd.DateOffset(months=12)
lista_m12 = [fecha_doce_meses_atras + pd.DateOffset(months=i) for i in range(12)]
lista_m12

fecha_seis_meses_atras = now - pd.DateOffset(months=6)
lista_m6 = [fecha_seis_meses_atras + pd.DateOffset(months=i) for i in range(6)]
lista_m6

[Timestamp('2025-09-01 00:00:00'),
 Timestamp('2025-10-01 00:00:00'),
 Timestamp('2025-11-01 00:00:00'),
 Timestamp('2025-12-01 00:00:00'),
 Timestamp('2026-01-01 00:00:00'),
 Timestamp('2026-02-01 00:00:00')]

In [193]:
temp_df_ventas = df_ventas[df_ventas["id_cliente"].isin(pan_rec["id_cliente"].unique())]

In [194]:
qw = (
    temp_df_ventas[["id_cliente", "mes"]]
    .drop_duplicates()
    .sort_values(["id_cliente", "mes"])
    .groupby("id_cliente")
    .tail(12)
    .reset_index(drop=True)
)
qw["mes"] = pd.to_datetime(qw["mes"])
qw["m12"] = qw["mes"].isin(lista_m12)
qw["m6"] = qw["mes"].isin(lista_m6)

In [195]:
categoria_cliente = qw.groupby("id_cliente")[["m12", "m6"]].sum().reset_index()

In [196]:
# Definir condiciones
condicion_super = categoria_cliente["m12"] == 12
condicion_frecuente = (
    (categoria_cliente["m12"] < 12)
    & (categoria_cliente["m12"] >= 6)
    & (categoria_cliente["m6"] == 6)
)
condicion_regulares = (categoria_cliente["m6"] <= 6) & (categoria_cliente["m12"] <= 6)
condicion_riesgo = (categoria_cliente["m6"] < 6) & (categoria_cliente["m6"] >= 4)
condicion_irregulares = categoria_cliente["m6"] < 4

# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_super, "categoria_cliente"] = "Super"
categoria_cliente.loc[condicion_frecuente, "categoria_cliente"] = "Frecuente"
categoria_cliente.loc[condicion_regulares, "categoria_cliente"] = "Regular"
categoria_cliente.loc[condicion_riesgo, "categoria_cliente"] = "Riesgo"
categoria_cliente.loc[condicion_irregulares, "categoria_cliente"] = "Irregular"

**Recalcular Irregularidad**

In [197]:
# Definir condiciones
condicion_regular = (
    (categoria_cliente["categoria_cliente"] == "Super")
    | (categoria_cliente["categoria_cliente"] == "Frecuente")
    | (categoria_cliente["categoria_cliente"] == "Regular")
    | (categoria_cliente["categoria_cliente"] == "Riesgo")
)
condicion_irregular = categoria_cliente["categoria_cliente"] == "Irregular"
# Aplicar condiciones y asignar categorías
categoria_cliente.loc[condicion_regular, "categoria_cliente_2"] = "Regular"
categoria_cliente.loc[condicion_irregular, "categoria_cliente_2"] = "Irregular"

In [198]:
categoria_cliente

,id_cliente,m12,m6,categoria_cliente,categoria_cliente_2
0,EC|0090|10003,11,6,Frecuente,Regular
1,EC|0090|10021,11,6,Frecuente,Regular
2,EC|0090|10037,11,6,Frecuente,Regular
3,EC|0090|10039,11,6,Frecuente,Regular
4,EC|0090|10119,11,6,Frecuente,Regular
...,...,...,...,...,...
18376,EC|0090|9965,11,6,Frecuente,Regular
18377,EC|0090|9974,11,6,Frecuente,Regular
18378,EC|0090|9976,11,6,Frecuente,Regular
18379,EC|0090|9996,11,6,Frecuente,Regular


In [199]:
categoria_cliente["categoria_cliente"].value_counts(dropna=False)

categoria_cliente
Frecuente    15515
Riesgo        1419
Irregular     1128
Super          173
Regular        146
Name: count, dtype: int64

In [200]:
categoria_cliente["categoria_cliente_2"].value_counts(dropna=False)

categoria_cliente_2
Regular      17253
Irregular     1128
Name: count, dtype: int64

## 5.1. Obteniendo marcas de recomendaciones por cliente

In [201]:
pan_rec = pd.read_csv("Output/D_rutas_rec.csv", sep=";")

In [202]:
pan_ventas = df_ventas[(df_ventas["cod_ruta"].isin(rutas))]
pan_ventas2 = df_ventas[(df_ventas["cod_ruta"].isin(low_sku_ruta))]
pan_ventas = pd.concat([pan_ventas, pan_ventas2], axis=0)

In [203]:
marca_articulo = pan_ventas[["desc_categoria", "cod_articulo_magic"]].drop_duplicates()
print(marca_articulo.shape)
marca_articulo.head()

(159, 2)


,desc_categoria,cod_articulo_magic
0,GASEOSAS,599653
1,CITRUS,509728
2,AGUA,508514
3,GASEOSAS,523079
4,CITRUS,511920


In [204]:
cliente_rec_marca = pd.merge(
    pan_rec, marca_articulo, on="cod_articulo_magic", how="left"
)
cliente_rec_marca["desc_categoria"] = cliente_rec_marca["desc_categoria"].str.strip()
print(cliente_rec_marca.shape)
cliente_rec_marca

(261770, 3)


,id_cliente,cod_articulo_magic,desc_categoria
0,EC|0090|10003,515408,BEBIDAS DEPORTIVAS
1,EC|0090|10003,508514,AGUA
2,EC|0090|10003,599211,CONSERVA DE PESCADO
3,EC|0090|10003,515912,BEBIDAS DEPORTIVAS
4,EC|0090|10003,524187,AGUA
...,...,...,...
261765,EC|0090|9999,514411,NECTAR
261766,EC|0090|9999,621870,CAFE
261767,EC|0090|9999,623423,DETERGENTE
261768,EC|0090|9999,523106,GASEOSAS


## 5.2. Calcular marcas distintas recomendadas por cliente

In [205]:
cods2 = cliente_rec_marca.groupby("id_cliente")["desc_categoria"].nunique().reset_index()
print(cods2.shape)
cods2.head()

(18381, 2)


,id_cliente,desc_categoria
0,EC|0090|10003,9
1,EC|0090|10021,11
2,EC|0090|10037,7
3,EC|0090|10039,9
4,EC|0090|10119,12


## 5.3 Quitando los SKUs de las ultimas 2 semanas (Evitar recompra)

In [206]:
last_2_weeks = pd.to_datetime(datetime.now(pytz.timezone("America/Lima"))) - pd.DateOffset(
    days=14
)
last_2_weeks

Timestamp('2026-03-16 15:48:22.905655-0500', tz='America/Lima')

In [207]:
# Asegurarse de que 'fecha_liquidacion' esté en formato datetime
pan_ventas["fecha_liquidacion"] = pd.to_datetime(pan_ventas["fecha_liquidacion"])
# Convertir 'fecha_liquidacion' a la misma zona horaria que 'last_year'
pan_ventas["fecha_liquidacion"] = pan_ventas["fecha_liquidacion"].dt.tz_localize(
    "America/Lima", nonexistent="NaT", ambiguous="NaT"
)

In [208]:
df_quitar_2_weeks = pan_ventas[pan_ventas["fecha_liquidacion"]>=last_2_weeks][["id_cliente","cod_articulo_magic"]].drop_duplicates().reset_index(drop=True)

In [209]:
# Hacemos un merge para identificar coincidencias
cliente_rec_sin4 = cliente_rec_marca.merge(df_quitar_2_weeks, on=['id_cliente', 'cod_articulo_magic'], how='left', indicator=True)
# Filtramos solo los registros que NO están en df_quitar_2_weeks
cliente_rec_sin4 = cliente_rec_sin4[cliente_rec_sin4['_merge'] == 'left_only'].drop(columns=['_merge'])

In [210]:
cliente_rec_marca.id_cliente.nunique()

18381

In [211]:
cliente_rec_sin4.id_cliente.nunique()

18092

In [212]:
cliente_rec_sin4

,id_cliente,cod_articulo_magic,desc_categoria
0,EC|0090|10003,515408,BEBIDAS DEPORTIVAS
2,EC|0090|10003,599211,CONSERVA DE PESCADO
3,EC|0090|10003,515912,BEBIDAS DEPORTIVAS
4,EC|0090|10003,524187,AGUA
5,EC|0090|10003,524189,AGUA
...,...,...,...
261764,EC|0090|9999,511920,CITRUS
261766,EC|0090|9999,621870,CAFE
261767,EC|0090|9999,623423,DETERGENTE
261768,EC|0090|9999,523106,GASEOSAS


## 5.4 Filtro para calcula antiguedad de clientes

In [213]:
pan_ventas = pan_ventas.rename(columns={"fecha_creacion_cliente": "fecha_creacion"})

In [214]:
# Convertir la columna 'fecha_creacion' a tipo fecha
pan_ventas["fecha_creacion"] = pd.to_datetime(
    pan_ventas["fecha_creacion"], format="%Y%m%d"
)

In [215]:
# Definir una función para etiquetar los clientes según su fecha de creación
def etiquetar_cliente(fecha_creacion):
    if pd.isnull(fecha_creacion):
        return (
            "nf"  # Si la fecha de creación es nula, etiquetar como 'nf' (no encontrada)
        )
    else:
        hoy = datetime.now()  # Obtener la fecha actual
        hace_12_meses = hoy - timedelta(days=365)  # Calcular hace 12 meses
        if fecha_creacion >= hace_12_meses:
            return "new_client"  # Si la fecha de creación está dentro de los últimos 12 meses, etiquetar como 'new_client'
        else:
            return "old_client"  # Si la fecha de creación es anterior a los últimos 12 meses, etiquetar como 'old_client'

In [216]:
# Aplicar la función a la columna 'fecha_creacion' para crear la nueva columna 'tipo_cliente'
pan_ventas["antiguedad"] = pan_ventas["fecha_creacion"].apply(etiquetar_cliente)

In [217]:
pan_antiguedad_clientes = (
    pan_ventas[["id_cliente", "fecha_creacion", "antiguedad"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

## 5.4.2 Obteniendo datos necesarios para armar DF para subir a SalesForce

In [218]:
# pan_ventas.groupby(["id_cliente","cod_compania","cod_sucursal","cod_cliente","cod_modulo"])[["cant_cajafisicavta","cant_cajaunitvta"]].sum().reset_index()

In [219]:
datos_para_salesforce = (
    pan_ventas.groupby(
        ["id_cliente", "cod_compania", "cod_sucursal", "cod_cliente", "cod_ruta"]
    )[
        [
            "id_cliente",
            "cod_compania",
            "cod_sucursal",
            "cod_cliente",
            "cod_modulo",
            "cod_ruta",
        ]
    ]
    .head(1)
    .reset_index(drop=True)
)

In [220]:
datos_para_salesforce["cod_cliente"]=datos_para_salesforce["cod_cliente"].astype(int)
datos_para_salesforce["cod_sucursal"]=datos_para_salesforce["cod_sucursal"].astype(int)
datos_para_salesforce["cod_compania"]=datos_para_salesforce["cod_compania"].astype(int)
datos_para_salesforce["cod_modulo"]=datos_para_salesforce["cod_modulo"].astype(int)
datos_para_salesforce["cod_ruta"]=datos_para_salesforce["cod_ruta"].astype(int)

## 5.5 Juntando las Recomendaciones Finales
 - Ordenado de acuerdo al peso de la recomendación y el id de usuario.
 - Hay 5 skus de 5 marcas distintas que se recomiendan a cada usuario
 - Ningún SKU que haya comprado el usuario en sus ultimas X visitas será recomendado. La cantidad de visitas (X) a ignorar depende de sus visitas promedio mensuales.

In [221]:
# Obtener primer SKU de cada marca de la recomendacion
final_rec = cliente_rec_sin4.groupby(["id_cliente", "desc_categoria"]).first().reset_index()
# Agregar Marca de sus primeras compras
final_rec = pd.merge(final_rec, cods2, how="left", on="id_cliente")
# Agregar descripcion de SKU recomendado
pan_prod = pan_prod[["cod_articulo_magic", "desc_articulo"]].drop_duplicates()
pan_prod = pan_prod.groupby(["cod_articulo_magic"]).first().reset_index()
final_rec = pd.merge(final_rec, pan_prod, how="left", on="cod_articulo_magic")
# Cambiar nombre de columnas
final_rec.columns = ["id_cliente", "marca_rec", "sku", "len_marca_rec", "desc_rec"]
# Agregar Giro del cliente
giros = df_ventas[df_ventas["id_cliente"].isin(final_rec["id_cliente"].unique())][
    ["id_cliente", "desc_giro", "desc_subgiro"]
].drop_duplicates()
final_rec = pd.merge(final_rec, giros, on="id_cliente", how="left")
# Agregar si el cliente es Regular o Irregular
final_rec = pd.merge(
    final_rec,
    categoria_cliente[["id_cliente", "categoria_cliente_2"]],
    on="id_cliente",
    how="left",
)
final_rec.loc[final_rec["categoria_cliente_2"] == "Irregular", "desc_giro"] = np.nan
# Agregar antiguedad de cliente
final_rec = pd.merge(final_rec, pan_antiguedad_clientes, how="left", on="id_cliente")
# Agregar segmento de cliente
info_segmentos = pan_ventas[
    [
        "id_cliente",
        "new_segment",
        "dias_de_visita__c",
        "periodo_de_visita__c",
        "ultima_visita",
    ]
].drop_duplicates()
info_segmentos = info_segmentos.groupby(["id_cliente"]).first().reset_index()
final_rec = pd.merge(final_rec, info_segmentos, how="left", on="id_cliente")
# Agregar datos necesarios para SALESFORCE
final_rec = pd.merge(final_rec, datos_para_salesforce, how="left", on="id_cliente")
# Agregar Peso de marcas dependiendo del giro del cliente
final_rec["peso"] = final_rec.apply(
    lambda row: mapeo_diccionario.get(row["desc_subgiro"], {}).get(row["marca_rec"], 5),
    axis=1,
)

In [222]:
# Ordenar la recomendacion de acuerdo al peso de la marca y nos quedamos con las 5 primeras marcas
final_rec = (
    final_rec.sort_values(["id_cliente", "peso"]).groupby(["id_cliente"]).head(5)
)
# Agregamos rank de la marca recomendada
final_rec["marca_rec_rank"] = final_rec.groupby("id_cliente").cumcount() + 1
final_rec = final_rec.reset_index(drop=True)

In [223]:
final_rec["sku"] = final_rec["sku"].astype(int)
final_rec["peso"] = final_rec["peso"].astype(int)
final_rec["marca_rec_rank"] = final_rec["marca_rec_rank"].astype(int)

In [224]:
final_rec

,id_cliente,marca_rec,sku,len_marca_rec,desc_rec,desc_giro,desc_subgiro,categoria_cliente_2,fecha_creacion,antiguedad,...,dias_de_visita__c,periodo_de_visita__c,ultima_visita,cod_compania,cod_sucursal,cod_cliente,cod_modulo,cod_ruta,peso,marca_rec_rank
0,EC|0090|10003,BEBIDAS DEPORTIVAS,515408,9,SPORADE UVA PET NO RETORNABLE 500 ML 12,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,10003,246,4917,1,1
1,EC|0090|10003,GASEOSAS,523108,9,BIG LIMON SIN AZUCAR PET NO RETORNABLE 300 ML 24,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,10003,246,4917,1,2
2,EC|0090|10003,AGUA,524187,9,CIELO MANZANA PET NO RETORNABLE 600 ML 6,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,10003,246,4917,2,3
3,EC|0090|10003,NECTAR,514411,9,PULP DURAZNO TETRA PAK 250 ml 24 pack,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,10003,246,4917,2,4
4,EC|0090|10003,DETERGENTE,623423,9,DESS DETERGENTE POLVO LIMON FUNDA 500 GRS 6,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,10003,246,4917,9,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76933,EC|0090|9999,GASEOSAS,500010,8,BIG COLA PET NO RETORNABLE 300 ML 24,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,9999,246,4917,1,1
76934,EC|0090|9999,CITRUS,511920,8,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,9999,246,4917,3,2
76935,EC|0090|9999,DETERGENTE,623423,8,DESS DETERGENTE POLVO LIMON FUNDA 500 GRS 6,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,9999,246,4917,9,3
76936,EC|0090|9999,CAFE,621870,8,D GUSSTO CAFE LIOFILIZADO STICK 6 ML 12 MC,BODEGA Y PARTICULARES,"BODEGAS, TIENDAS",Regular,NaT,nf,...,2;5,F1,2026-03-27,90,17,9999,246,4917,10,4


## 5.7 Filtro de Recomendaciones por Segmento de cliente

In [225]:
# Definir una función para filtrar las filas según el valor de 'new_segment'
def filtrar_segmento(group):
    segmento = group["new_segment"].iloc[
        0
    ]  # Obtener el valor del segmento para el grupo
    if segmento == "BLINDAR":
        return group.head(1)  # Si el segmento es 'Blindar', mantener la primera fila
    elif segmento == "MANTENER":
        return group.head(
            2
        )  # Si el segmento es 'Mantener', mantener las dos primeras filas
    elif segmento == "DESARROLLAR":
        return group.head(
            3
        )  # Si el segmento es 'Desarrollar', mantener las tres primeras filas
    elif segmento == "OPTIMIZAR":
        return group.head(
            4
        )  # Si el segmento es 'Optimizar', mantener las cuatro primeras filas
    else:
        return group  # Si el segmento no es ninguno de los especificados, mantener todas las filas

In [226]:
final_rec = (
    final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)
)

/tmp/ipykernel_31118/3325175710.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_rec.groupby("id_cliente").apply(filtrar_segmento).reset_index(drop=True)


In [227]:
final_rec.groupby("new_segment")["id_cliente"].nunique()

new_segment
BLINDAR        2850
DESARROLLAR    4981
MANTENER       6235
OPTIMIZAR      4026
Name: id_cliente, dtype: int64

In [228]:
final_rec.groupby(["new_segment","id_cliente"]).sku.nunique().groupby("new_segment").mean()

new_segment
BLINDAR        1.000000
DESARROLLAR    2.869303
MANTENER       1.956215
OPTIMIZAR      3.119722
Name: sku, dtype: float64

## 5.8. Guardando recomendacion para D&A y Comercial

In [229]:
final_rec.id_cliente.nunique()

18092

In [230]:
final_rec.dias_de_visita__c.value_counts(dropna=False)

dias_de_visita__c
2;5              41110
2                  788
1;2;3;4;5;6;7        1
Name: count, dtype: int64

In [231]:
final_rec.periodo_de_visita__c.value_counts(dropna=False)

periodo_de_visita__c
F1    41899
Name: count, dtype: int64

In [232]:
final_rec.ultima_visita.value_counts(dropna=False)

ultima_visita
2026-03-27    40573
2026-03-24      747
2026-03-20       66
2026-03-17       61
2026-03-13       61
2026-01-13       55
2026-01-20       42
2026-02-24       40
2026-03-06       36
2026-03-10       35
2026-01-23       27
2026-01-30       21
2026-03-03       20
2026-01-16       19
2026-01-09       16
2026-02-20       15
2026-02-06       14
2026-02-03       13
2026-02-13       11
2026-02-10        8
2026-02-17        4
2026-02-27        3
2026-01-02        3
2026-01-27        3
2026-01-06        3
2026-03-23        2
2026-03-28        1
Name: count, dtype: int64

### 5.8.1 Para D&A

In [233]:
fecha_tomorrow = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=1)
).strftime("%Y-%m-%d")
fecha_tomorrow

'2026-03-31'

In [234]:
final_rec.to_csv(
    f"Output/PS_piloto_data_v1/D_pan_recs_data_{fecha_tomorrow}.csv", index=False
)

In [235]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Ecuador/Output/PS_piloto_data_v1/D_pan_recs_data_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(final_rec, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Ecuador/Output/PS_piloto_data_v1/D_pan_recs_data_2026-03-31.csv'],
 'partitions_values': {}}

### 5.8.2 Para subir a PS

In [236]:
recomendaciones_para_salesforce = final_rec[
    ["cod_compania", "cod_sucursal", "cod_cliente", "cod_modulo", "sku"]
]
recomendaciones_para_salesforce["Pais"] = "EC"
recomendaciones_para_salesforce["Cajas"] = int(1)
recomendaciones_para_salesforce["Unidades"] = int(0)
recomendaciones_para_salesforce["Fecha"] = fecha_tomorrow
recomendaciones_para_salesforce = recomendaciones_para_salesforce[
    [
        "Pais",
        "cod_compania",
        "cod_sucursal",
        "cod_cliente",
        "cod_modulo",
        "sku",
        "Cajas",
        "Unidades",
        "Fecha",
    ]
]
recomendaciones_para_salesforce.columns = [
    "Pais",
    "Compania",
    "Sucursal",
    "Cliente",
    "Modulo",
    "Producto",
    "Cajas",
    "Unidades",
    "Fecha",
]

/tmp/ipykernel_31118/4041848949.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recomendaciones_para_salesforce["Pais"] = "EC"
/tmp/ipykernel_31118/4041848949.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recomendaciones_para_salesforce["Cajas"] = int(1)
/tmp/ipykernel_31118/4041848949.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.

In [237]:
recomendaciones_para_salesforce["Compania"] = recomendaciones_para_salesforce[
    "Compania"
].apply(lambda x: str(int(x)).rjust(4, "0"))
recomendaciones_para_salesforce["Sucursal"] = recomendaciones_para_salesforce[
    "Sucursal"
].apply(lambda x: str(int(x)).rjust(2, "0"))

In [238]:
recomendaciones_para_salesforce.Sucursal.unique()

array(['17', '11', '04', '03', '07', '06', '08', '09'], dtype=object)

In [239]:
recomendaciones_para_salesforce.Compania.unique()

array(['0090'], dtype=object)

In [240]:
recomendaciones_para_salesforce.Cliente.nunique()

18092

In [241]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
0,EC,0090,17,10003,246,515408,1,0,2026-03-31
1,EC,0090,17,10003,246,523108,1,0,2026-03-31
2,EC,0090,17,10021,246,515417,1,0,2026-03-31
3,EC,0090,17,10037,195,515519,1,0,2026-03-31
4,EC,0090,17,10037,195,524189,1,0,2026-03-31
...,...,...,...,...,...,...,...,...,...
41894,EC,0090,17,9996,246,621870,1,0,2026-03-31
41895,EC,0090,17,9996,246,623423,1,0,2026-03-31
41896,EC,0090,17,9999,246,500010,1,0,2026-03-31
41897,EC,0090,17,9999,246,511920,1,0,2026-03-31


In [242]:
recomendaciones_para_salesforce.Cliente.nunique()

18092

In [243]:
rutas_cliente = df_ventas[["cod_cliente", "cod_ruta"]].drop_duplicates()

**Conteo de marcas recomendadas**

In [244]:
# count = (
#     final_rec.groupby("id_cliente")["marca_rec"]
#     .apply(lambda x: "-".join((x)))
#     .reset_index(name="marcas_ordenadas")
# )
# count = (
#     count.groupby("marcas_ordenadas")["id_cliente"]
#     .nunique()
#     .sort_values(ascending=False)
# )
# count

**Conteo de SKUs recomendadas**

In [245]:
# count2 = (
#     final_rec.groupby("id_cliente")["sku"]
#     .apply(lambda x: "-".join(x.astype(str)))
#     .reset_index(name="marcas_ordenadas")
# )
# count2 = (
#     count2.groupby("marcas_ordenadas")["id_cliente"]
#     .nunique()
#     .sort_values(ascending=False)
#     .reset_index()
# )
# count2["f"] = count2["id_cliente"] / count2["id_cliente"].sum()
# count2["fa"] = count2["f"].cumsum()
# count2

In [246]:
# print(f"RESUMEN para {fecha_tomorrow}")
# print(
#     "Total de clientes a recomendar: ",
#     recomendaciones_para_salesforce.Cliente.nunique(),
# )
# print("Combinaciones de recomendaciones de MARCAS:", count.shape[0])
# print("Marcar recomendadas:", final_rec.marca_rec.unique())
# print("Combinaciones de recomendaciones de SKUs:", count2.shape[0])
# print(
#     "Combinaciones de recomendaciones de SKUs al 80% de clientes:",
#     count2[count2["fa"] <= 0.8].shape[0],
# )
# print("SKUs usados en la recomendacion:", final_rec["sku"].nunique())

In [247]:
recomendaciones_para_salesforce

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha
0,EC,0090,17,10003,246,515408,1,0,2026-03-31
1,EC,0090,17,10003,246,523108,1,0,2026-03-31
2,EC,0090,17,10021,246,515417,1,0,2026-03-31
3,EC,0090,17,10037,195,515519,1,0,2026-03-31
4,EC,0090,17,10037,195,524189,1,0,2026-03-31
...,...,...,...,...,...,...,...,...,...
41894,EC,0090,17,9996,246,621870,1,0,2026-03-31
41895,EC,0090,17,9996,246,623423,1,0,2026-03-31
41896,EC,0090,17,9999,246,500010,1,0,2026-03-31
41897,EC,0090,17,9999,246,511920,1,0,2026-03-31


In [248]:
recomendaciones_para_salesforce.groupby("Cliente")["Producto"].nunique().mean()

2.315885474242759

In [249]:
asd = (
    recomendaciones_para_salesforce.groupby("Cliente")
    .agg({"Producto": ["count", "nunique"]})
    .reset_index()
)
asd.columns = ["Cliente", "count", "nunique"]
asd["dif"] = asd["count"] == asd["nunique"]
asd.sort_values(["dif", "count"])

,Cliente,count,nunique,dif
0,1953,1,1,True
5,2112,1,1,True
8,2130,1,1,True
10,2172,1,1,True
18,2848,1,1,True
...,...,...,...,...
16601,480742,4,4,True
17226,483337,4,4,True
17284,483538,4,4,True
17437,484252,4,4,True


In [250]:
asd.dif.value_counts()

dif
True    18092
Name: count, dtype: int64

In [251]:
recomendaciones_para_salesforce.to_csv(
    f"Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv", index=False
)

In [252]:
# Establecer la conexión con S3
bucket_name = 'aje-analytics-ps-backup'  # nombre de bucket en S3
file_name = f'PS_Ecuador/Output/PS_piloto_v1/D_base_pedidos_{fecha_tomorrow}.csv'  # nombre para el archivo en S3
s3_path = f's3://{bucket_name}/{file_name}'

# Escribir el dataframe en S3 con AWS Data Wrangler
wr.s3.to_csv(recomendaciones_para_salesforce, s3_path, index=False)

{'paths': ['s3://aje-analytics-ps-backup/PS_Ecuador/Output/PS_piloto_v1/D_base_pedidos_2026-03-31.csv'],
 'partitions_values': {}}

In [253]:
recomendaciones_para_salesforce.shape

(41899, 9)

In [254]:
pan_ventas.cod_compania=pan_ventas.cod_compania.astype(int)

In [255]:
pan_ventas["cod_compania"] = (
    pan_ventas["cod_compania"].astype(str).apply(lambda x: str(int(x)).rjust(4, "0"))
)

In [256]:
recomendaciones_para_salesforce.dtypes

Pais        object
Compania    object
Sucursal    object
Cliente      int64
Modulo       int64
Producto     int64
Cajas        int64
Unidades     int64
Fecha       object
dtype: object

In [257]:
pan_ventas["cod_cliente"] = pan_ventas["cod_cliente"].astype(int)

In [258]:
val = pd.merge(recomendaciones_para_salesforce,pan_prod,left_on="Producto",right_on="cod_articulo_magic",how="left")
val = pd.merge(val,pan_ventas[["cod_compania","cod_cliente","cod_ruta","desc_sucursal","fecha_proceso"]].drop_duplicates(),left_on=["Compania","Cliente"],right_on=["cod_compania","cod_cliente"],how="left")

In [259]:
val.cod_cliente = val.cod_cliente.astype(int)
val.cod_ruta = val.cod_ruta.astype(int)
val.fecha_proceso = val.fecha_proceso.astype(int)

In [260]:
val

,Pais,Compania,Sucursal,Cliente,Modulo,Producto,Cajas,Unidades,Fecha,cod_articulo_magic,desc_articulo,cod_compania,cod_cliente,cod_ruta,desc_sucursal,fecha_proceso
0,EC,0090,17,10003,246,515408,1,0,2026-03-31,515408,SPORADE UVA PET NO RETORNABLE 500 ML 12,0090,10003,4917,CEDIS GUAYAQUIL SUR,20260330
1,EC,0090,17,10003,246,523108,1,0,2026-03-31,523108,BIG LIMON SIN AZUCAR PET NO RETORNABLE 300 ML 24,0090,10003,4917,CEDIS GUAYAQUIL SUR,20260330
2,EC,0090,17,10021,246,515417,1,0,2026-03-31,515417,SPORADE UVA PET NO RETORNABLE 1200 ML 6,0090,10021,4917,CEDIS GUAYAQUIL SUR,20260330
3,EC,0090,17,10037,195,515519,1,0,2026-03-31,515519,SPORADE FRUTAS TROPICALES PET NO RETORNABLE 12...,0090,10037,3117,CEDIS GUAYAQUIL SUR,20260330
4,EC,0090,17,10037,195,524189,1,0,2026-03-31,524189,CIELO LIMONADA PET NO RETORNABLE 600 ML 6,0090,10037,3117,CEDIS GUAYAQUIL SUR,20260330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41895,EC,0090,17,9996,246,621870,1,0,2026-03-31,621870,D GUSSTO CAFE LIOFILIZADO STICK 6 ML 12 MC,0090,9996,4917,CEDIS GUAYAQUIL SUR,20260330
41896,EC,0090,17,9996,246,623423,1,0,2026-03-31,623423,DESS DETERGENTE POLVO LIMON FUNDA 500 GRS 6,0090,9996,4917,CEDIS GUAYAQUIL SUR,20260330
41897,EC,0090,17,9999,246,500010,1,0,2026-03-31,500010,BIG COLA PET NO RETORNABLE 300 ML 24,0090,9999,4917,CEDIS GUAYAQUIL SUR,20260330
41898,EC,0090,17,9999,246,511920,1,0,2026-03-31,511920,"CIFRUT NARANJA, MANDARINA Y LIMON PET NO RETOR...",0090,9999,4917,CEDIS GUAYAQUIL SUR,20260330


In [261]:
fecha_hoy_val = (
    datetime.now(pytz.timezone("America/Lima")) + timedelta(days=0)
).strftime("%Y-%m-%d")
fecha_hoy_val

'2026-03-30'

In [262]:
# val.to_csv(f"Val_comercial/D_EC_validacion_recomendaciones_{fecha_hoy_val}.csv",index=False)

In [263]:
# # Detener el monitoreo
# running = False
# monitor_thread.join()

# # Graficar consumo de RAM y CPU
# fig, ax1 = plt.subplots(figsize=(10, 5))

# ax1.set_xlabel("Tiempo (segundos)")
# ax1.set_ylabel("RAM usada (GB)", color="tab:blue")
# ax1.plot(timestamps, mem_usage_gb, marker="o", linestyle="-", color="tab:blue", label="RAM (GB)")
# ax1.tick_params(axis="y", labelcolor="tab:blue")

# ax2 = ax1.twinx()  # Crear segundo eje Y para CPU
# ax2.set_ylabel("CPU usada (%)", color="tab:red")
# ax2.plot(timestamps, cpu_usage_percent, marker="x", linestyle="--", color="tab:red", label="CPU (%)")
# ax2.tick_params(axis="y", labelcolor="tab:red")

# fig.suptitle("Consumo de CPU y RAM del Notebook")
# ax1.grid()
# fig.tight_layout()
# plt.show()

In [264]:
# Apaga el Kernel para luego poder usar los demás notebooks sin llenar la memoria
import IPython
app = IPython.get_ipython()
app.kernel.do_shutdown(False)  # True indica apagar completamente el kernel


{'status': 'ok', 'restart': False}